# **Sistema de recomendación híbrido para H&M con aprendizaje no supervisado**

Este proyecto construye y evalúa un sistema de recomendación de productos de moda a partir de la base *H&M Personalized Fashion Recommendations*, publicada en Kaggle por el grupo H&M. La base registra las compras de 1.371.980 clientes sobre un catálogo de 105.542 artículos entre el 20 de septiembre de 2018 y el 22 de septiembre de 2020, con un total de 31.788.324 transacciones. El propósito es comparar, sobre un mismo problema y bajo un mismo esquema de evaluación, los métodos de aprendizaje no supervisado y de sistemas de recomendación, y combinarlos en un recomendador híbrido.

### **1. Planteamiento del problema**

#### **1.1 Contexto**

En el comercio minorista de moda, el catálogo es amplio y cambia con rapidez: cada temporada entran y salen productos, y un cliente solo llega a conocer una fracción mínima de la oferta. Los sistemas de recomendación buscan reducir ese espacio de búsqueda proponiendo a cada cliente un conjunto pequeño de productos con alta probabilidad de interés. A diferencia de los conjuntos clásicos con calificaciones explícitas (MovieLens), en este caso solo se observan **compras**: se sabe qué adquirió cada cliente, pero no qué le disgustó ni qué vio sin comprar. Se trata, por tanto, de un problema de **retroalimentación implícita**, en el que la ausencia de compra no equivale a rechazo (Hu, Koren y Volinsky, 2008).

#### **1.2 Datos**

La base está compuesta por cuatro tablas en formato *parquet*:

- **`transactions_train`**: una fila por artículo comprado, con fecha (`t_dat`), cliente (`customer_id`), artículo (`article_id`), precio normalizado (`price`, sin unidad monetaria) y canal de venta (`sales_channel_id`).
- **`customers`**: atributos de cada cliente, como edad (`age`), estado de membresía (`club_member_status`), frecuencia de recepción de noticias (`fashion_news_frequency`) y código postal anonimizado.
- **`articles`**: catálogo con 25 atributos jerárquicos (tipo de producto, grupo, color, departamento, sección, grupo de prenda) y una descripción textual (`detail_desc`).
- **`sample_submission`**: formato de entrega del concurso original, con doce artículos recomendados por cliente; no se utiliza en el análisis.

Las recomendaciones se generan a nivel de **`product_code`**, que agrupa las variantes de color de un mismo producto (`product_code = article_id // 1000`). Esto reduce el catálogo de 105.542 artículos a 47.224 productos y conserva la información sobre el tipo de prenda que el cliente prefiere.

#### **1.3 Pregunta de investigación**

¿En qué medida un recomendador híbrido, que combina la segmentación no supervisada de clientes con filtrado colaborativo, recomendación basada en contenido y reglas de asociación, mejora la recomendación de productos frente a una línea base de popularidad, y qué fortalezas y debilidades presenta cada método en este contexto?

#### **1.4 Objetivos**

**Objetivo general.** Diseñar, implementar y evaluar un sistema de recomendación híbrido para la base de H&M, comparando su desempeño con el de los métodos individuales que lo componen.

**Objetivos específicos.**

1. Seleccionar una muestra estratificada de clientes que preserve la estructura de la población y sea computacionalmente manejable.
2. Construir perfiles de comportamiento de compra y reducir su dimensión mediante componentes principales.
3. Segmentar a los clientes con K-medias, K-medoides, clustering jerárquico, mezclas gaussianas y DBSCAN, y comparar la calidad y estabilidad de las particiones.
4. Implementar recomendadores de popularidad, por segmento, por reglas de asociación, de filtrado colaborativo (similitud de coseno y SVD), basados en contenido (TF-IDF) y con máquinas de Boltzmann restringidas.
5. Combinar los recomendadores individuales en un modelo híbrido ponderado, calibrado en una semana de validación.
6. Evaluar todos los modelos sobre una semana de prueba posterior y discutir su eficacia e idoneidad para el problema.

#### **1.5 Enfoque metodológico**

La evaluación respeta el orden temporal de las compras: los modelos se entrenan con la historia hasta el 8 de septiembre de 2020, se calibran con la semana del 9 al 15 de septiembre y se evalúan con la semana del 16 al 22 de septiembre. Esta partición evita que información futura se filtre al entrenamiento, un problema que puede inflar las métricas y alterar el orden entre modelos (Ji et al., 2023). Para cada cliente evaluado se recomiendan doce productos, y el desempeño se mide con métricas de *ranking* (Precisión@12, Recall@12 y MAP@12), más adecuadas que el error cuadrático medio cuando la tarea es recomendar una lista corta de productos (Cremonesi, Koren y Turrin, 2010).

Todo modelo se compara contra una línea base de popularidad, pues la literatura muestra que métodos simples bien ajustados superan con frecuencia a modelos más complejos (Dacrema, Cremonesi y Jannach, 2019). El análisis se realiza sobre una muestra estratificada del 10 % de los clientes, cuya viabilidad computacional se evalúa en la sección 2 y cuyo diseño estadístico se desarrolla en la sección 3.

### **2. Carga de datos y diagnóstico de viabilidad computacional**

La base de transacciones contiene más de 31 millones de registros, por lo que no se carga completa en memoria. Se consulta directamente sobre los archivos *parquet* con **DuckDB**, que procesa los datos por bloques y devuelve solo resultados agregados; a `pandas` pasan únicamente las tablas resumen.

El diagnóstico verifica que `product_code` se obtiene como `article_id // 1000` (los tres últimos dígitos identifican la variante de color), define las ventanas temporales —la última semana como **prueba**, la anterior como **validación** y el período restante como **entrenamiento**—, cuantifica el catálogo de candidatos (productos vendidos en las 12 semanas previas a la validación) y estima, para fracciones de muestreo del 2 %, 5 %, 10 % y 20 %, el número de clientes, transacciones y compradores evaluables, junto con la memoria requerida por la matriz dispersa cliente × producto y por la matriz de puntuaciones.

In [1]:
import os, zipfile, subprocess
import pyarrow.parquet as pq

DS = "derrickmwiti/hm-personalized-fashion-recommendations"
archivos = ["articles.parquet", "customers.parquet",
            "sample_submission.parquet", "transactions_train.parquet"]
os.makedirs("hm", exist_ok=True)

for f in archivos:
    ruta = f"hm/{f}"
    if not os.path.exists(ruta):
        subprocess.run(["kaggle", "datasets", "download", DS, "-f", f, "-p", "hm"], check=True)
        z = f"{ruta}.zip"
        if os.path.exists(z):              # a veces llega comprimido
            zipfile.ZipFile(z).extractall("hm"); os.remove(z)

    pf = pq.ParquetFile(ruta)
    print("="*70, f"\n{f} | filas: {pf.metadata.num_rows:,} | row groups: {pf.num_row_groups}")
    print(pf.schema_arrow)
    display(next(pf.iter_batches(batch_size=5)).to_pandas())

KeyboardInterrupt: 

In [ ]:
# ============================================================
# 2. CARGA DE DATOS Y DIAGNÓSTICO DE VIABILIDAD
# ============================================================

!pip install -q duckdb psutil

import duckdb
import numpy as np
import pandas as pd
import psutil

RUTA = "hm"                                    # Carpeta con los parquet descargados
TX = f"'{RUTA}/transactions_train.parquet'"
CU = f"'{RUTA}/customers.parquet'"
AR = f"'{RUTA}/articles.parquet'"

con = duckdb.connect()

# Memoria disponible en la sesión de Colab
mem = psutil.virtual_memory()
print(f"RAM total: {mem.total/1e9:.1f} GB | RAM disponible: {mem.available/1e9:.1f} GB")

# ------------------------------------------------------------
# 2.1 Verificación de product_code = article_id // 1000
# ------------------------------------------------------------
verif = con.sql(f"""
    SELECT COUNT(*) AS articulos,
           SUM(CASE WHEN article_id // 1000 = product_code THEN 1 ELSE 0 END) AS coinciden,
           COUNT(DISTINCT product_code) AS product_codes
    FROM {AR}
""").df()
print("\nVerificación de product_code:")
print(verif)

# Vista de transacciones con fecha y product_code
con.sql(f"""
    CREATE OR REPLACE TEMP VIEW tx AS
    SELECT CAST(t_dat AS DATE)   AS fecha,
           customer_id,
           article_id // 1000    AS product_code
    FROM {TX}
""")

# ------------------------------------------------------------
# 2.2 Dimensiones generales
# ------------------------------------------------------------
gen = con.sql("""
    SELECT MIN(fecha) AS inicio, MAX(fecha) AS fin,
           COUNT(*) AS transacciones,
           COUNT(DISTINCT customer_id) AS clientes_con_compras,
           COUNT(DISTINCT product_code) AS product_codes_vendidos
    FROM tx
""").df()
N_marco = con.sql(f"SELECT COUNT(*) FROM {CU}").fetchone()[0]
print(f"\nClientes en el marco (customers): {N_marco:,}")
print(gen)

# ------------------------------------------------------------
# 2.3 Ventanas temporales: entrenamiento | validación | prueba
# ------------------------------------------------------------
fin = pd.Timestamp(gen.loc[0, "fin"])
corte_test = fin - pd.Timedelta(days=7)          # prueba: últimos 7 días
corte_val  = fin - pd.Timedelta(days=14)         # validación: 7 días anteriores
ini_cand   = corte_val - pd.Timedelta(weeks=12)  # ventana de candidatos

print(f"\nEntrenamiento: {gen.loc[0,'inicio']} → {corte_val.date()}")
print(f"Validación:    {(corte_val + pd.Timedelta(days=1)).date()} → {corte_test.date()}")
print(f"Prueba:        {(corte_test + pd.Timedelta(days=1)).date()} → {fin.date()}")

vent = con.sql(f"""
    SELECT
      COUNT(DISTINCT CASE WHEN fecha <= DATE '{corte_val.date()}' THEN customer_id END)
          AS clientes_entrenamiento,
      COUNT(DISTINCT CASE WHEN fecha >  DATE '{corte_val.date()}'
                           AND fecha <= DATE '{corte_test.date()}' THEN customer_id END)
          AS compradores_validacion,
      COUNT(DISTINCT CASE WHEN fecha >  DATE '{corte_test.date()}' THEN customer_id END)
          AS compradores_prueba,
      COUNT(DISTINCT CASE WHEN fecha >  DATE '{ini_cand.date()}'
                           AND fecha <= DATE '{corte_val.date()}' THEN product_code END)
          AS candidatos_12_semanas
    FROM tx
""").df()
print("\nPoblación por ventana:")
print(vent.T)

# Pares únicos cliente × product_code en entrenamiento (entradas no nulas de la matriz)
nnz = con.sql(f"""
    SELECT COUNT(*) FROM (
        SELECT DISTINCT customer_id, product_code
        FROM tx WHERE fecha <= DATE '{corte_val.date()}'
    )
""").fetchone()[0]
print(f"\nPares cliente × product_code en entrenamiento: {nnz:,}")

# ------------------------------------------------------------
# 2.4 Escenarios por fracción de muestreo (valores esperados)
# ------------------------------------------------------------
n_tx    = gen.loc[0, "transacciones"]
n_test  = vent.loc[0, "compradores_prueba"]
n_val   = vent.loc[0, "compradores_validacion"]
n_train = vent.loc[0, "clientes_entrenamiento"]
n_cand  = vent.loc[0, "candidatos_12_semanas"]

escenarios = []
for f in [0.02, 0.05, 0.10, 0.20]:
    escenarios.append({
        "fracción": f"{f:.0%}",
        "clientes": round(f * N_marco),
        "transacciones": round(f * n_tx),
        "clientes_entrenamiento": round(f * n_train),
        "compradores_validación": round(f * n_val),
        "compradores_prueba": round(f * n_test),
        # CSR float32: 4 B (dato) + 4 B (índice de columna) por entrada no nula
        "matriz_dispersa_MB": round(f * nnz * 8 / 1e6, 1),
        # Puntuaciones densas float32: compradores de prueba × candidatos
        "puntuaciones_prueba_MB": round(f * n_test * n_cand * 4 / 1e6, 1),
    })

pd.DataFrame(escenarios)

### **Análisis del diagnóstico de viabilidad**

La verificación confirma que `product_code` corresponde exactamente a `article_id // 1000` en los 105.542 artículos, por lo que el nivel de recomendación se reduce de 105.542 artículos a 47.224 productos, de los cuales 46.834 registran ventas. La base cubre del 20 de septiembre de 2018 al 22 de septiembre de 2020, con 31.788.324 transacciones de 1.362.281 clientes; solo 9.699 clientes del marco (0,7 %) no registran compras.

La partición temporal deja la semana del 16 al 22 de septiembre de 2020 como prueba (68.984 compradores) y la semana anterior como validación (72.019 compradores). Ambas ventanas tienen tamaños similares, lo que respalda el uso de la validación para calibrar el modelo híbrido. Además, solo 20.639 productos (44 % del catálogo) se vendieron en las 12 semanas previas a la validación, lo que evidencia la alta rotación del catálogo de moda y justifica restringir el conjunto de candidatos.

En términos computacionales, la memoria no constituye una restricción: aun con una fracción del 20 %, la matriz dispersa cliente × producto ocupa 38,5 MB y la matriz de puntuaciones de los compradores de prueba 1,1 GB, frente a 12,3 GB disponibles. La restricción relevante es el número de compradores en la semana de prueba, del cual depende la precisión con que se comparan los modelos, y el tiempo de entrenamiento, que crece proporcionalmente con el número de clientes. Una fracción del 10 % ofrece cerca de 6.900 compradores evaluables y aproximadamente 3,2 millones de transacciones, un equilibrio razonable entre precisión de la evaluación y costo computacional; esta elección se valida estadísticamente en el diseño muestral de la sección 3.

### **3. Diseño muestral estratificado**

El análisis se realiza sobre una muestra de clientes y no sobre la base completa. La **unidad de muestreo es el cliente**: al seleccionarlo se conservan todas sus transacciones, de modo que cada cliente muestreado aporta su historia de compra completa, que es la información que requieren los recomendadores. Muestrear transacciones de forma individual fragmentaría esas historias. El diseño sigue el esquema de muestreo estratificado con selección aleatoria simple sin reemplazo dentro de cada estrato (Särndal, Swensson y Wretman, 1992).

#### **3.1 Marco muestral y variables de estratificación**

El marco muestral está formado por los 1.371.980 clientes de la tabla `customers`. Se estratifica por el cruce de dos variables:

- **Nivel de actividad**: número de transacciones del cliente, agrupado en 0–2, 3–9, 10–29, 30–99 y 100 o más. La actividad de compra es muy asimétrica: pocos clientes concentran una proporción alta de las transacciones, y una muestra sin estratificar podría subrepresentarlos. Los clientes sin compras (0,7 % del marco, sección 2) se fusionan con el grupo de 1–2 compras para evitar un estrato marginal.
- **Grupo de edad**: menores de 25, 25–34, 35–49, 50 o más, y una categoría **N/D** para los clientes sin edad registrada. La edad se asocia con las preferencias de moda y es la variable demográfica que se usará para recomendar a clientes sin historial.

El cruce genera hasta 25 estratos. La actividad se calcula **solo con las transacciones del período de entrenamiento** (hasta el 8 de septiembre de 2020): si se incluyeran las semanas de validación y prueba, la probabilidad de selección de un cliente dependería de sus compras futuras, que son justamente las que se pretende predecir.

#### **3.2 Diagnóstico de los estratos**

Para cada estrato se calcula su tamaño ($N_h$), su peso en la población ($W_h = N_h / N$) y la media, la desviación estándar ($S_h$) y el coeficiente de variación ($CV_h$) del número de compras, que se toma como variable de estudio. Estos insumos permiten evaluar la homogeneidad de los estratos y calcular las afijaciones en las subsecciones siguientes.

In [ ]:
# ============================================================
# 3. DISEÑO MUESTRAL ESTRATIFICADO
# ============================================================

import matplotlib.pyplot as plt

SEED = 2026                                  # Semilla de reproducibilidad
FRACCION = 0.10                              # Fracción definida en la sección 2
CORTE_ENTRENAMIENTO = corte_val.date()       # Último día de entrenamiento (sección 2)

# ------------------------------------------------------------
# 3.1 Marco muestral y variables de estratificación
# ------------------------------------------------------------

# Actividad de cada cliente en el período de entrenamiento (calculada en DuckDB)
marco = con.sql(f"""
    WITH act AS (
        SELECT customer_id,
               COUNT(*)                            AS n_compras,
               SUM(price)                          AS gasto,
               COUNT(DISTINCT article_id // 1000)  AS n_productos
        FROM {TX}
        WHERE CAST(t_dat AS DATE) <= DATE '{CORTE_ENTRENAMIENTO}'
        GROUP BY customer_id
    )
    SELECT c.customer_id,
           c.age,
           COALESCE(a.n_compras, 0)   AS n_compras,
           COALESCE(a.gasto, 0)       AS gasto,
           COALESCE(a.n_productos, 0) AS n_productos
    FROM {CU} c
    LEFT JOIN act a USING (customer_id)
""").df()

N = len(marco)
print(f"Clientes en el marco: {N:,}")
print(f"Clientes sin edad registrada: {marco['age'].isna().sum():,} "
      f"({100 * marco['age'].isna().mean():.1f} %)")
print("\nDistribución del número de compras en entrenamiento:")
print(marco["n_compras"].describe(percentiles=[.25, .5, .75, .9, .99]).round(2))

# Estrato de actividad (0 compras fusionado con 1–2)
marco["estrato_act"] = pd.cut(
    marco["n_compras"],
    bins=[-1, 2, 9, 29, 99, np.inf],
    labels=["0–2", "3–9", "10–29", "30–99", "100+"]
)

# Estrato de edad, con categoría N/D para faltantes
marco["estrato_edad"] = pd.cut(
    marco["age"],
    bins=[0, 24, 34, 49, np.inf],
    labels=["<25", "25–34", "35–49", "50+"]
).cat.add_categories("N/D").fillna("N/D")

# Estrato final: actividad × edad
marco["estrato"] = marco["estrato_act"].astype(str) + " | " + marco["estrato_edad"].astype(str)

# ------------------------------------------------------------
# 3.2 Diagnóstico de los estratos
# ------------------------------------------------------------

# Tamaño de los estratos en tabla cruzada
print("\nClientes por estrato (actividad × edad):")
print(pd.crosstab(marco["estrato_act"], marco["estrato_edad"], margins=True, margins_name="Total"))

# Parámetros por estrato para la variable de estudio (número de compras)
estratos_tbl = (
    marco.groupby(["estrato", "estrato_act", "estrato_edad"], observed=True)
         .agg(N_h=("customer_id", "size"),
              media_y_h=("n_compras", "mean"),
              S_h=("n_compras", "std"),
              total_y_h=("n_compras", "sum"))
         .reset_index()
)
estratos_tbl["S_h"]  = estratos_tbl["S_h"].fillna(0)
estratos_tbl["W_h"]  = estratos_tbl["N_h"] / N                                 # Peso del estrato
estratos_tbl["CV_h"] = 100 * estratos_tbl["S_h"] / estratos_tbl["media_y_h"]   # Heterogeneidad relativa
estratos_tbl["part_compras"] = 100 * estratos_tbl["total_y_h"] / estratos_tbl["total_y_h"].sum()

print(f"\nNúmero de estratos: {len(estratos_tbl)}")

# Visualización: tamaño y coeficiente de variación por estrato
orden = estratos_tbl.sort_values("N_h")
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].barh(orden["estrato"], orden["N_h"], color="steelblue")
axes[0].set_title("Tamaño de los estratos ($N_h$)")
axes[0].set_xlabel("Número de clientes")

axes[1].barh(orden["estrato"], orden["CV_h"], color="darkgreen")
axes[1].set_title("Coeficiente de variación del número de compras ($CV_h$)")
axes[1].set_xlabel("CV (%)")

plt.tight_layout()
plt.show()

estratos_tbl.sort_values(["estrato_act", "estrato_edad"]).round(3)

**Distribución de la actividad.** El número de compras en el período de entrenamiento tiene media 22,81 y mediana 9, con desviación estándar 38,60 (coeficiente de variación global cercano al 169 %). La distribución presenta una fuerte asimetría a la derecha: el 25 % de los clientes compra a lo sumo 3 veces, el 90 % no supera 58 compras y el 99 % no supera 184, pero el máximo llega a 1.876 transacciones. Un volumen de ese orden en dos años es atípico para un consumidor individual; podría corresponder a compras para terceros o a reventa, aunque esta explicación es una hipótesis que no puede verificarse con los datos disponibles.

**Concentración de las compras.** Los 55.767 clientes del grupo 100+ representan el 4,1 % del marco, pero concentran el 29,2 % de las compras. Si se agregan los grupos 30–99 y 100+, el 22,4 % de los clientes acumula el 71,3 % de las transacciones. En el extremo opuesto, el grupo 0–2 reúne el 20,2 % de los clientes y apenas el 1,2 % de las compras. Esta concentración justifica estratificar por actividad: una muestra aleatoria simple podría sub- o sobrerrepresentar a los clientes más activos, que son los que aportan la mayor parte de la información de compra.

**Composición por edad.** Solo el 1,2 % de los clientes no tiene edad registrada. Los grupos de edad tienen pesos similares en el marco (<25: 26,0 %; 25–34: 28,7 %; 35–49: 21,0 %; 50+: 23,2 %), pero su composición cambia con la actividad: el grupo 25–34 representa el 42,3 % de los clientes con 100 o más compras, frente al 28,7 % del total, mientras que los menores de 25 pasan del 26,0 % al 14,2 %.

**Homogeneidad dentro de los estratos.** Dentro de cada nivel de actividad, la media de compras es prácticamente igual entre grupos de edad (por ejemplo, entre 16,5 y 17,7 en el nivel 10–29). Los coeficientes de variación por estrato se ubican entre el 32 % y el 50 %, muy por debajo del 169 % global, lo que indica que la estratificación produce grupos internamente mucho más homogéneos que la población. La desviación estándar crece con el nivel de actividad (de cerca de 0,6 en 0–2 a entre 68 y 84 en 100+), lo que anticipa que una afijación de Neyman concentrará la muestra en los estratos de mayor actividad.

**Interpretación.** La reducción de la variabilidad proviene casi por completo de la actividad; la edad aporta poco a la precisión para el número de compras. Su papel en el diseño es otro: garantizar la representación de cada grupo de edad en la muestra, pues la edad se usará para caracterizar segmentos y para recomendar a clientes sin historial. La contribución de cada variable se cuantifica en la descomposición de la subsección siguiente.

**Estratos pequeños.** Los estratos con edad N/D son los de menor tamaño. Con una afijación proporcional del 10 %, el estrato 100+ | N/D (60 clientes) aportaría solo 6 clientes y el estrato 30–99 | N/D (898 clientes) cerca de 90. Para evitar un estrato con tan pocas unidades se fusionan ambos en un estrato **30+ | N/D** de 958 clientes, con lo que el diseño queda con 24 estratos.

In [ ]:
# ============================================================
# 3.3 DESCOMPOSICIÓN DE LA VARIABILIDAD Y 3.4 AFIJACIONES
# ============================================================

FUSIONAR_ND_ALTOS = True    # Fusiona 30–99 | N/D y 100+ | N/D en "30+ | N/D"

if FUSIONAR_ND_ALTOS:
    mask = (marco["estrato_edad"] == "N/D") & marco["estrato_act"].isin(["30–99", "100+"])
    marco.loc[mask, "estrato"] = "30+ | N/D"

# Parámetros por estrato (recalculados tras la fusión)
estratos_tbl = (
    marco.groupby("estrato")
         .agg(N_h=("customer_id", "size"),
              media_y_h=("n_compras", "mean"),
              S_h=("n_compras", "std"))
         .reset_index()
)
estratos_tbl["S_h"] = estratos_tbl["S_h"].fillna(0)
estratos_tbl["W_h"] = estratos_tbl["N_h"] / N
estratos_tbl["grupo_act"] = estratos_tbl["estrato"].str.split(" | ", regex=False).str[0]
H = len(estratos_tbl)
print(f"Número de estratos: {H}")

# ------------------------------------------------------------
# 3.3 Descomposición SST = SSW + SSB
# ------------------------------------------------------------
def descomponer_varianza(df, y, estrato):
    """Descompone la suma de cuadrados total de y en dentro (SSW) y entre (SSB) estratos."""
    media = df[y].mean()
    sst = ((df[y] - media) ** 2).sum()
    medias_h = df.groupby(estrato, observed=True)[y].transform("mean")
    ssw = ((df[y] - medias_h) ** 2).sum()
    ssb = ((medias_h - media) ** 2).sum()
    return pd.DataFrame({
        "estratificación": estrato,
        "componente": ["SST total", "SSW dentro", "SSB entre"],
        "valor": [sst, ssw, ssb],
        "proporción": [1.0, ssw / sst, ssb / sst],
    })

descomp = pd.concat([
    descomponer_varianza(marco, "n_compras", "estrato_act"),
    descomponer_varianza(marco, "n_compras", "estrato_edad"),
    descomponer_varianza(marco, "n_compras", "estrato"),
])
print("\nDescomposición de la variabilidad del número de compras:")
print(descomp.round(4).to_string(index=False))

# ------------------------------------------------------------
# 3.4 Estrategias de afijación
# ------------------------------------------------------------
n_total = int(round(FRACCION * N))

def ajustar_afijacion(raw, N_h, n_total, n_min=2):
    """Redondea la afijación, respeta 2 <= n_h <= N_h y completa n_total por mayor residuo."""
    n_h = np.minimum(np.maximum(np.floor(raw), n_min), N_h).astype(int)
    while n_h.sum() < n_total:                       # Faltan unidades
        libres = np.where(n_h < N_h)[0]
        orden = libres[np.argsort(-(raw[libres] - n_h[libres]))]
        n_h[orden[: n_total - n_h.sum()]] += 1
    while n_h.sum() > n_total:                       # Sobran unidades
        n_h[np.argmax(np.where(n_h > n_min, n_h, -1))] -= 1
    return n_h

def tabla_afijacion(tbl, n_total, metodo, alpha=0.5):
    """Calcula n_h, f_h, pesos y contribución a la varianza para un método de afijación."""
    N_h = tbl["N_h"].to_numpy()
    S_h = np.where(tbl["S_h"].to_numpy() == 0, 1, tbl["S_h"].to_numpy())
    if metodo == "Proporcional":
        raw = n_total * N_h / N_h.sum()
    elif metodo == "Uniforme":
        raw = np.full(len(N_h), n_total / len(N_h))
    elif metodo == "Neyman":
        raw = n_total * N_h * S_h / (N_h * S_h).sum()
    elif metodo == "Potencia":
        raw = n_total * N_h * S_h**alpha / (N_h * S_h**alpha).sum()
    out = tbl[["estrato", "grupo_act", "N_h", "S_h", "W_h"]].copy()
    out["método"] = metodo
    out["n_h"] = ajustar_afijacion(raw.astype(float), N_h, n_total)
    out["f_h"] = out["n_h"] / out["N_h"]
    out["w_h"] = 1 / out["f_h"]
    out["var_h"] = out["W_h"]**2 * (1 - out["f_h"]) * out["S_h"]**2 / out["n_h"]
    return out

metodos = ["Proporcional", "Uniforme", "Neyman", "Potencia"]
afijaciones = pd.concat([tabla_afijacion(estratos_tbl, n_total, m) for m in metodos])

# Varianza teórica bajo MAS del mismo tamaño
S2 = marco["n_compras"].var()
var_mas = (1 - n_total / N) * S2 / n_total

resumen_afij = (
    afijaciones.groupby("método", sort=False)
               .agg(n_total=("n_h", "sum"),
                    min_n_h=("n_h", "min"), max_n_h=("n_h", "max"),
                    min_f_h=("f_h", "min"), max_f_h=("f_h", "max"),
                    min_w=("w_h", "min"), max_w=("w_h", "max"),
                    var_teorica=("var_h", "sum"))
               .reset_index()
)
resumen_afij["DEFF_teorico"] = resumen_afij["var_teorica"] / var_mas
resumen_afij["CV_estimador_%"] = 100 * np.sqrt(resumen_afij["var_teorica"]) / marco["n_compras"].mean()
print(f"\nTamaño muestral total: n = {n_total:,} | Varianza MAS: {var_mas:.6f}")
print(resumen_afij.round(4).to_string(index=False))

# Distribución de la muestra por nivel de actividad según el método
dist_act = (afijaciones.groupby(["método", "grupo_act"], sort=False)["n_h"].sum()
                       .unstack("grupo_act"))
dist_act = dist_act[["0–2", "3–9", "10–29", "30–99", "100+", "30+"]]
dist_act_pct = 100 * dist_act.div(dist_act.sum(axis=1), axis=0)
pob_act = 100 * estratos_tbl.groupby("grupo_act")["N_h"].sum() / N
dist_act_pct.loc["Población"] = pob_act[dist_act_pct.columns]
print("\nDistribución porcentual de la muestra por nivel de actividad:")
print(dist_act_pct.round(1))

# Gráfico: participación de cada nivel de actividad en la muestra
dist_act_pct.T.plot(kind="bar", figsize=(11, 5), width=0.8)
plt.title("Participación de cada nivel de actividad en la muestra según la afijación")
plt.xlabel("Nivel de actividad")
plt.ylabel("Porcentaje de la muestra (%)")
plt.xticks(rotation=0)
plt.legend(title="Método")
plt.tight_layout()
plt.show()

**Aporte de cada variable.** La actividad explica por sí sola el 77,95 % de la variabilidad del número de compras (SSB), mientras que la edad explica apenas el 1,41 %. El cruce actividad × edad alcanza el 77,99 %, es decir, la edad añade solo 0,04 puntos porcentuales sobre la actividad. Esto confirma lo observado en el diagnóstico: la ganancia de eficiencia proviene casi exclusivamente de la actividad, y la edad se conserva en el diseño por su papel en la caracterización de los segmentos y en la recomendación a clientes sin historial, no por su aporte a la precisión.

**Eficiencia de las afijaciones.** Las cuatro estrategias mejoran de forma marcada frente a un MAS del mismo tamaño. La afijación de Neyman alcanza el menor efecto de diseño ($DEFF = 0{,}034$), seguida de la de potencia (0,060), la uniforme (0,090) y la proporcional (0,220). El $DEFF$ de la afijación proporcional coincide con la proporción $SSW/SST$ (0,2201), resultado esperado por teoría: con fracciones de muestreo iguales, la varianza del estimador estratificado depende solo de la variabilidad dentro de los estratos. A diferencia de lo que suele ocurrir, la afijación uniforme resulta aquí más eficiente que la proporcional, porque al asignar el mismo tamaño a todos los estratos sobremuestrea los de mayor actividad, que son también los de mayor desviación estándar, y se acerca así a la lógica de Neyman.

**Composición de la muestra.** La ganancia de eficiencia de Neyman se obtiene a costa de distorsionar fuertemente la composición de la muestra: los clientes con 100 o más compras pasarían del 4,1 % de la población al 36,5 % de la muestra, y los del grupo 0–2 del 20,2 % al 1,4 %. Esto exige pesos de expansión entre 1,05 y 160,75, con un estrato prácticamente censado ($f_h = 0{,}95$). La afijación de potencia suaviza esta distorsión (pesos entre 2,59 y 31,97), y la proporcional reproduce exactamente la composición poblacional, con pesos constantes cercanos a 10.

**Decisión.** Se adopta la **afijación proporcional**. El objetivo del muestreo en este proyecto no es estimar el número medio de compras, sino obtener una muestra sobre la cual entrenar y evaluar recomendadores. Estos modelos no incorporan pesos de expansión: las frecuencias de popularidad, las similitudes entre clientes, la estructura de los segmentos y las propias métricas de evaluación se calculan sobre la muestra tal como es. Con Neyman o potencia, los clientes más activos dominarían todos esos cálculos y los resultados no representarían a la población. La muestra proporcional es autoponderada, de modo que los modelos pueden entrenarse sin ponderar y sin sesgar la composición, y conserva una ganancia de eficiencia considerable: una reducción del 78 % de la varianza frente al MAS, con un coeficiente de variación del estimador de la media del 0,20 %.

#### **3.5 Evaluación Monte Carlo de los diseños**

La varianza teórica de la subsección anterior se contrasta con el comportamiento empírico de los diseños. Se extraen $B = 300$ muestras con cada afijación y con un MAS del mismo tamaño, y en cada réplica se estima la media del número de compras mediante el estimador de Horvitz-Thompson, $\hat{\bar{y}} = \frac{1}{N}\sum_{k \in s} w_k\, y_k$, con $w_k = N_h / n_h$. A partir de las réplicas se calculan el sesgo relativo, el error estándar empírico, la raíz del error cuadrático medio, el coeficiente de variación y el $DEFF$ empírico. El ejercicio se repite para el **gasto total** del cliente, variable que no intervino en la afijación, con el fin de verificar si la ganancia de eficiencia se extiende a variables no utilizadas en el diseño.

#### **3.6 Selección de la muestra y verificación**

Se extrae la muestra definitiva con afijación proporcional y selección aleatoria simple sin reemplazo dentro de cada estrato. Se verifica que reproduzca la composición de la población en las variables de estratificación y en variables no utilizadas para estratificar (estado de membresía y frecuencia de noticias), y se cuantifica el número de compradores de la muestra en las semanas de validación y prueba. La lista de clientes seleccionados se guarda en un archivo *parquet* para las secciones siguientes.

In [ ]:
# ============================================================
# 3.5 EVALUACIÓN MONTE CARLO DE LOS DISEÑOS
# ============================================================

B = 300
rng = np.random.default_rng(SEED)

# Arreglos para muestreo rápido con numpy
y_compras = marco["n_compras"].to_numpy(dtype=float)
y_gasto   = marco["gasto"].to_numpy(dtype=float)
codigos   = marco["estrato"].astype("category")
idx_por_estrato = {e: np.where(codigos == e)[0] for e in codigos.cat.categories}

media_real = {"n_compras": y_compras.mean(), "gasto": y_gasto.mean()}

def muestra_estratificada(n_h_dict, rng):
    """Selección MAS sin reemplazo dentro de cada estrato; devuelve índices y pesos HT."""
    idx, w = [], []
    for e, n_h in n_h_dict.items():
        pool = idx_por_estrato[e]
        sel = rng.choice(pool, size=n_h, replace=False)
        idx.append(sel)
        w.append(np.full(n_h, len(pool) / n_h))
    return np.concatenate(idx), np.concatenate(w)

def muestra_mas(n, rng):
    """MAS sin reemplazo sobre todo el marco."""
    return rng.choice(N, size=n, replace=False), np.full(n, N / n)

disenos = {"MAS": None}
for m in metodos:
    tab = afijaciones[afijaciones["método"] == m]
    disenos[m] = dict(zip(tab["estrato"], tab["n_h"]))

registros = []
for nombre, n_h_dict in disenos.items():
    for b in range(B):
        if n_h_dict is None:
            idx, w = muestra_mas(n_total, rng)
        else:
            idx, w = muestra_estratificada(n_h_dict, rng)
        registros.append({
            "diseño": nombre, "réplica": b,
            "n_compras": (w * y_compras[idx]).sum() / N,   # Media HT
            "gasto":     (w * y_gasto[idx]).sum() / N,
        })

sim = pd.DataFrame(registros)

# Resumen del desempeño empírico por diseño y variable
filas = []
for var in ["n_compras", "gasto"]:
    var_mas_emp = sim.loc[sim["diseño"] == "MAS", var].var()
    for nombre, g in sim.groupby("diseño", sort=False):
        est = g[var]
        sesgo = est.mean() - media_real[var]
        filas.append({
            "variable": var, "diseño": nombre,
            "media_real": media_real[var],
            "media_estimada": est.mean(),
            "sesgo_rel_%": 100 * sesgo / media_real[var],
            "ee_empírico": est.std(),
            "RECM": np.sqrt(((est - media_real[var]) ** 2).mean()),
            "CV_%": 100 * est.std() / est.mean(),
            "DEFF_empírico": est.var() / var_mas_emp,
        })
tabla_mc = pd.DataFrame(filas)
print(tabla_mc.round(5).to_string(index=False))

# Distribuciones Monte Carlo del estimador por diseño
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, var, titulo in zip(axes, ["n_compras", "gasto"],
                           ["Media del número de compras", "Media del gasto"]):
    for nombre, g in sim.groupby("diseño", sort=False):
        ax.hist(g[var], bins=30, alpha=0.45, label=nombre)
    ax.axvline(media_real[var], color="black", linestyle="--", label="Valor poblacional")
    ax.set_title(f"Distribución Monte Carlo: {titulo.lower()}")
    ax.set_xlabel(titulo)
    ax.set_ylabel("Frecuencia")
    ax.legend()
plt.tight_layout()
plt.show()

**Sesgo.** Todos los diseños producen estimaciones centradas en el valor poblacional: los sesgos relativos son inferiores al 0,03 % en valor absoluto para ambas variables, lo que confirma el comportamiento insesgado del estimador de Horvitz-Thompson bajo cada esquema. Los histogramas muestran el mismo resultado: todas las distribuciones se centran sobre la línea del valor poblacional y difieren solo en su dispersión.

**Precisión para el número de compras.** El MAS presenta el mayor error estándar empírico (0,0981) y un coeficiente de variación del 0,43 %. La afijación proporcional lo reduce a la mitad (0,0484; CV de 0,21 %), y las afijaciones uniforme, de potencia y de Neyman lo reducen aún más (0,0274, 0,0263 y 0,0185, respectivamente). Los $DEFF$ empíricos (proporcional 0,243; uniforme 0,078; potencia 0,072; Neyman 0,035) son consistentes con los teóricos de la subsección 3.4 (0,220; 0,090; 0,060 y 0,034); las diferencias son las esperables al estimar una varianza con 300 réplicas, y el orden de los diseños se mantiene salvo entre uniforme y potencia, cuyos valores son muy cercanos.

**Precisión para el gasto.** La ganancia de eficiencia se extiende al gasto total, variable que no intervino en la afijación. La afijación proporcional reduce la varianza un 63 % frente al MAS ($DEFF = 0{,}370$), y Neyman un 94 % ($DEFF = 0{,}061$). La ganancia es menor que para el número de compras porque la estratificación se construyó sobre la actividad, pero sigue siendo considerable: el gasto está fuertemente asociado con el número de compras, de modo que los estratos de actividad también son homogéneos en gasto.

**Conclusión.** La simulación confirma empíricamente lo obtenido en la teoría: la estratificación por actividad es eficiente, y Neyman sería el diseño óptimo si el objetivo fuera estimar medias poblacionales. Para el propósito de este proyecto se mantiene la afijación proporcional, por las razones expuestas en la subsección 3.4: es autoponderada y reduce la varianza entre un 63 % y un 76 % frente al MAS en las dos variables evaluadas.

In [ ]:
# ============================================================
# 3.6 SELECCIÓN DE LA MUESTRA Y VERIFICACIÓN
# ============================================================

# Muestra definitiva: afijación proporcional
rng_final = np.random.default_rng(SEED)
idx_muestra, w_muestra = muestra_estratificada(disenos["Proporcional"], rng_final)

muestra = marco.iloc[idx_muestra].copy()
muestra["w"] = w_muestra
print(f"Clientes en la muestra: {len(muestra):,} | "
      f"Peso mínimo: {muestra['w'].min():.4f} | Peso máximo: {muestra['w'].max():.4f}")

# Variables no usadas en la estratificación
extra = con.sql(f"""
    SELECT customer_id, club_member_status, fashion_news_frequency
    FROM {CU}
""").df()
marco_ext   = marco[["customer_id", "estrato_act", "estrato_edad"]].merge(extra, on="customer_id")
muestra_ext = muestra[["customer_id", "estrato_act", "estrato_edad"]].merge(extra, on="customer_id")

def comparar(var):
    """Distribución porcentual de una variable en la población y en la muestra."""
    pob = marco_ext[var].astype("object").fillna("N/D").value_counts(normalize=True) * 100
    mue = muestra_ext[var].astype("object").fillna("N/D").value_counts(normalize=True) * 100
    t = pd.DataFrame({"población_%": pob, "muestra_%": mue}).fillna(0)
    t["diferencia_pp"] = t["muestra_%"] - t["población_%"]
    return t.round(3)

for var in ["estrato_act", "estrato_edad", "club_member_status", "fashion_news_frequency"]:
    print(f"\n{var}:")
    print(comparar(var))

# Compradores de la muestra en validación y prueba
compradores = con.sql(f"""
    SELECT DISTINCT customer_id,
           CASE WHEN fecha > DATE '{corte_test.date()}' THEN 'prueba' ELSE 'validación' END AS ventana
    FROM tx
    WHERE fecha > DATE '{corte_val.date()}'
""").df()
comp_muestra = compradores[compradores["customer_id"].isin(muestra["customer_id"])]
print("\nCompradores de la muestra por ventana:")
print(comp_muestra["ventana"].value_counts())

# Guardar la muestra y registrarla en DuckDB para las secciones siguientes
muestra[["customer_id", "estrato", "estrato_act", "estrato_edad", "age", "w"]].to_parquet(
    f"{RUTA}/muestra_clientes.parquet", index=False)
con.sql(f"CREATE OR REPLACE TEMP VIEW muestra AS SELECT * FROM '{RUTA}/muestra_clientes.parquet'")
print(f"\nMuestra guardada en {RUTA}/muestra_clientes.parquet")

**Composición en las variables de estratificación.** La muestra contiene 137.198 clientes con pesos de expansión entre 9,98 y 10,01, lo que confirma su carácter autoponderado. Las distribuciones por nivel de actividad y por grupo de edad coinciden con las de la población hasta la tercera cifra decimal (diferencias de a lo sumo 0,001 puntos porcentuales), resultado esperado porque la afijación proporcional fija por construcción la participación de cada estrato.

**Composición en variables no utilizadas para estratificar.** La verificación relevante es la de variables que no intervinieron en el diseño. En el estado de membresía, la mayor diferencia es de 0,09 puntos porcentuales (PRE-CREATE: 6,78 % en la población frente a 6,87 % en la muestra), y en la frecuencia de noticias es de 0,16 puntos porcentuales (NONE: 63,97 % frente a 64,14 %). Ambas diferencias son pequeñas y compatibles con la variación aleatoria propia del muestreo, de modo que la muestra también reproduce la estructura poblacional en variables externas al diseño. Se observa además una inconsistencia de codificación en `fashion_news_frequency`, que contiene las categorías `NONE` y `None` como valores distintos; la segunda tiene una frecuencia despreciable y se unificará con la primera en la preparación de los perfiles.

**Compradores evaluables.** La muestra incluye 7.209 compradores en la semana de validación y 6.896 en la semana de prueba, prácticamente iguales a los valores esperados del diagnóstico de la sección 2 (7.202 y 6.898). Con ello se cuenta con cerca de 6.900 clientes sobre los cuales evaluar y comparar los recomendadores.

**Conclusión.** La muestra estratificada proporcional del 10 % reproduce la composición de la población tanto en las variables de diseño como en variables externas, no requiere ponderación en los modelos y ofrece un número suficiente de compradores para la evaluación. Queda guardada en `hm/muestra_clientes.parquet` y registrada como vista en DuckDB para las secciones siguientes.

### **4. Partición temporal y conjunto de candidatos**

#### **4.1 Transacciones de la muestra**

Se extraen de la base completa todas las transacciones de los 137.198 clientes muestreados. Para reducir el uso de memoria, el identificador alfanumérico de cada cliente se reemplaza por un índice entero y las variables numéricas se almacenan en tipos de menor tamaño. Las transacciones de la muestra se guardan en un archivo *parquet* para no repetir la extracción en las secciones siguientes.

#### **4.2 Esquema de validación temporal**

La evaluación se organiza en dos escenarios que respetan el orden cronológico:

- **Escenario de validación**: los modelos se entrenan con la historia hasta el 8 de septiembre de 2020 y se evalúan con las compras del 9 al 15 de septiembre. En este escenario se ajustan los hiperparámetros y los pesos del modelo híbrido.
- **Escenario de prueba**: con la configuración ya fijada, los modelos se reentrenan incluyendo la semana de validación (historia hasta el 15 de septiembre) y se evalúan con las compras del 16 al 22 de septiembre.

El reentrenamiento evita que quede una semana sin información entre lo que el modelo aprende y lo que predice, aspecto relevante dada la alta rotación del catálogo, y la semana de prueba no interviene en ninguna decisión de modelado.

#### **4.3 Conjunto de candidatos y techo alcanzable**

En cada escenario, los productos candidatos a recomendar son los vendidos en las 12 semanas previas al inicio de la ventana evaluada. Como los recomendadores solo pueden proponer productos del conjunto de candidatos, la proporción de compras evaluadas que pertenecen a dicho conjunto define el **techo** del Recall que puede alcanzar cualquier modelo. Se cuantifican, además, tres características de las compras evaluadas que orientan el diseño de los recomendadores: la proporción de **recompras** (productos que el cliente ya había comprado antes), la proporción de **productos nuevos** (sin ventas antes de la ventana evaluada) y la proporción de **compradores sin historia** previa.

In [ ]:
# ============================================================
# 4. PARTICIÓN TEMPORAL Y CONJUNTO DE CANDIDATOS
# ============================================================

# ------------------------------------------------------------
# 4.1 Transacciones de la muestra
# ------------------------------------------------------------
tx_m = con.sql(f"""
    SELECT CAST(t.t_dat AS DATE)      AS fecha,
           t.customer_id,
           t.article_id,
           t.article_id // 1000       AS product_code,
           t.price,
           t.sales_channel_id
    FROM {TX} t
    JOIN muestra m USING (customer_id)
""").df()

# Índice entero por cliente (ahorra memoria frente al hash de 64 caracteres)
clientes = muestra["customer_id"].reset_index(drop=True)
cliente_idx = pd.Series(np.arange(len(clientes), dtype=np.int32), index=clientes)
tx_m["cliente"] = tx_m["customer_id"].map(cliente_idx).astype(np.int32)
tx_m = tx_m.drop(columns="customer_id")

# Tipos compactos
tx_m["fecha"] = pd.to_datetime(tx_m["fecha"])
tx_m = tx_m.astype({"article_id": np.int32, "product_code": np.int32,
                    "price": np.float32, "sales_channel_id": np.int8})

print(f"Transacciones de la muestra: {len(tx_m):,}")
print(f"Clientes con al menos una transacción: {tx_m['cliente'].nunique():,} de {len(clientes):,}")
print(f"Memoria en uso: {tx_m.memory_usage(deep=True).sum() / 1e6:.1f} MB")

tx_m.to_parquet(f"{RUTA}/tx_muestra.parquet", index=False)
clientes.to_frame().assign(cliente=np.arange(len(clientes))).to_parquet(
    f"{RUTA}/clientes_idx.parquet", index=False)

# ------------------------------------------------------------
# 4.2 Esquema de validación temporal
# ------------------------------------------------------------
escenarios = {
    "Validación": {"fin_historia": corte_val,  "fin_objetivo": corte_test},
    "Prueba":     {"fin_historia": corte_test, "fin_objetivo": fin},
}

for nombre, esc in escenarios.items():
    print(f"{nombre}: historia hasta {esc['fin_historia'].date()} | "
          f"objetivo {(esc['fin_historia'] + pd.Timedelta(days=1)).date()} → {esc['fin_objetivo'].date()}")

# ------------------------------------------------------------
# 4.3 Conjunto de candidatos y techo alcanzable
# ------------------------------------------------------------
def candidatos_poblacion(fin_historia, semanas=12):
    """Productos vendidos en la población en las 'semanas' previas a fin_historia."""
    ini = fin_historia - pd.Timedelta(weeks=semanas)
    return set(con.sql(f"""
        SELECT DISTINCT product_code FROM tx
        WHERE fecha > DATE '{ini.date()}' AND fecha <= DATE '{fin_historia.date()}'
    """).df()["product_code"])

def vendidos_antes(fin_historia):
    """Productos con al menos una venta en la población hasta fin_historia."""
    return set(con.sql(f"""
        SELECT DISTINCT product_code FROM tx WHERE fecha <= DATE '{fin_historia.date()}'
    """).df()["product_code"])

filas = []
for nombre, esc in escenarios.items():
    hist = tx_m[tx_m["fecha"] <= esc["fin_historia"]]
    obj  = tx_m[(tx_m["fecha"] > esc["fin_historia"]) & (tx_m["fecha"] <= esc["fin_objetivo"])]

    cand = candidatos_poblacion(esc["fin_historia"])
    previos = vendidos_antes(esc["fin_historia"])
    esc["candidatos"] = cand                                   # Se conserva para las secciones siguientes

    pares_obj  = obj[["cliente", "product_code"]].drop_duplicates()
    pares_hist = set(zip(hist["cliente"], hist["product_code"]))
    clientes_hist = set(hist["cliente"])

    en_cand  = pares_obj["product_code"].isin(cand)
    nuevos   = ~pares_obj["product_code"].isin(previos)
    recompra = pd.Series([p in pares_hist for p in zip(pares_obj["cliente"], pares_obj["product_code"])],
                         index=pares_obj.index)
    prod_por_cliente = pares_obj.groupby("cliente").size()
    compradores = pares_obj["cliente"].unique()

    filas.append({
        "escenario": nombre,
        "candidatos": len(cand),
        "compradores": len(compradores),
        "pares_cliente_producto": len(pares_obj),
        "productos_por_comprador_media": prod_por_cliente.mean(),
        "productos_por_comprador_mediana": prod_por_cliente.median(),
        "techo_recall_%": 100 * en_cand.mean(),
        "recompras_%": 100 * recompra.mean(),
        "productos_nuevos_%": 100 * nuevos.mean(),
        "compradores_sin_historia_%": 100 * np.mean([c not in clientes_hist for c in compradores]),
        "compradores_con_alguna_compra_en_candidatos_%":
            100 * pares_obj[en_cand]["cliente"].nunique() / len(compradores),
    })

diag_ventanas = pd.DataFrame(filas).set_index("escenario").T
diag_ventanas.round(2)

**Transacciones de la muestra.** La muestra contiene 3.184.013 transacciones, cifra muy cercana a la esperada en el diagnóstico de la sección 2 (3.178.832), y ocupa 79,6 MB en memoria gracias al índice entero de clientes y a los tipos compactos. De los 137.198 clientes muestreados, 915 (0,7 %) no registran ninguna transacción, proporción igual a la observada en el marco.

**Techo alcanzable.** El conjunto de candidatos (20.639 productos en validación y 20.572 en prueba) contiene el 97,25 % y el 97,07 % de las compras evaluadas, respectivamente. Restringir los candidatos a los productos vendidos en las últimas 12 semanas reduce el catálogo a menos de la mitad y sacrifica menos del 3 % del Recall alcanzable. La pérdida corresponde casi por completo a productos nuevos sin ventas previas (2,61 % y 2,85 %), que ningún recomendador basado en historial de compras puede proponer.

**Recompra.** Solo el 8,12 % de los pares cliente–producto de validación y el 8,38 % de los de prueba corresponden a productos que el cliente ya había comprado. Al trabajar a nivel de `product_code`, una recompra implica volver a adquirir el mismo producto en cualquier color, de modo que más del 90 % de lo comprado en cada semana es nuevo para el cliente. En consecuencia, un recomendador basado únicamente en la recompra tendría un Recall máximo cercano al 8 %, y el problema central es anticipar productos que el cliente aún no ha adquirido.

**Compradores sin historia.** Entre el 7,48 % y el 8,47 % de los compradores evaluados no tienen compras previas. Para ellos no son aplicables los métodos basados en historial individual (filtrado colaborativo, contenido o RBM), por lo que requieren recomendaciones por popularidad o por segmento demográfico, lo que respalda el uso de la edad en el diseño.

**Tamaño de las canastas.** Cada comprador adquiere en promedio 2,97 productos distintos en validación y 2,84 en prueba, con una mediana de 2. Con listas de 12 recomendaciones, la Precisión@12 promedio no puede superar aproximadamente 0,25 aun con un recomendador perfecto. Por ello, los valores absolutos de precisión serán bajos por construcción y la comparación entre modelos debe hacerse en términos relativos, con el MAP@12 y el Recall@12 como métricas principales.

### **5. Análisis exploratorio**

El análisis exploratorio se realiza exclusivamente sobre las transacciones de la muestra en el período de entrenamiento (hasta el 8 de septiembre de 2020), de modo que ninguna decisión de modelado se apoye en información de las semanas de validación o prueba. Se examinan cinco aspectos relevantes para el diseño de los recomendadores:

- **Evolución temporal** de las transacciones y de los clientes activos por semana, para identificar estacionalidad y tendencia.
- **Canal de venta**, con la participación de las compras en tienda física y en línea.
- **Concentración de la demanda** entre productos, mediante la curva de Lorenz de las ventas. Una demanda muy concentrada favorece a los recomendadores de popularidad y advierte sobre el sesgo hacia productos populares.
- **Categorías** de producto más vendidas, a partir de los atributos de la tabla `articles`.
- **Frecuencia de compra** de los clientes, medida como el número de días entre fechas de compra consecutivas.

Antes de unir los atributos de producto se verifica que las variables descriptivas sean constantes dentro de cada `product_code`, condición necesaria para trabajar a ese nivel.

In [ ]:
# ============================================================
# 5. ANÁLISIS EXPLORATORIO
# ============================================================

tr = tx_m[tx_m["fecha"] <= corte_val]          # Solo período de entrenamiento
print(f"Transacciones de entrenamiento en la muestra: {len(tr):,}")

# Atributos de producto
art = con.sql(f"SELECT * FROM {AR}").df()
attrs = ["prod_name", "product_type_name", "product_group_name", "index_group_name",
         "index_name", "section_name", "garment_group_name", "detail_desc"]

# Verificación: atributos constantes dentro de cada product_code
variacion = {a: (art.groupby("product_code")[a].nunique(dropna=False) > 1).mean() * 100 for a in attrs}
print("\nPorcentaje de product_code con más de un valor por atributo:")
print(pd.Series(variacion).round(2))

prod = art.drop_duplicates("product_code").set_index("product_code")[attrs]

# ------------------------------------------------------------
# 5.1 Evolución temporal
# ------------------------------------------------------------
semanal = tr.groupby(pd.Grouper(key="fecha", freq="W")).agg(
    transacciones=("cliente", "size"), clientes_activos=("cliente", "nunique"))

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
axes[0].plot(semanal.index, semanal["transacciones"], color="steelblue")
axes[0].set_title("Transacciones por semana (muestra, entrenamiento)")
axes[0].set_ylabel("Transacciones")
axes[1].plot(semanal.index, semanal["clientes_activos"], color="darkorange")
axes[1].set_title("Clientes activos por semana")
axes[1].set_ylabel("Clientes")
axes[1].set_xlabel("Semana")
plt.tight_layout()
plt.show()
print(semanal.describe().round(1))

# ------------------------------------------------------------
# 5.2 Canal de venta
# ------------------------------------------------------------
canal = tr["sales_channel_id"].map({1: "Tienda", 2: "En línea"})
print("\nParticipación por canal (%):")
print((canal.value_counts(normalize=True) * 100).round(2))

canal_sem = (tr.assign(canal=canal)
               .groupby([pd.Grouper(key="fecha", freq="W"), "canal"]).size()
               .unstack().fillna(0))
canal_sem_pct = 100 * canal_sem.div(canal_sem.sum(axis=1), axis=0)
canal_sem_pct.plot(figsize=(13, 4), color=["darkgreen", "gray"])
plt.title("Participación semanal por canal de venta")
plt.ylabel("Porcentaje de transacciones (%)")
plt.xlabel("Semana")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 5.3 Concentración de la demanda
# ------------------------------------------------------------
ventas_prod = tr["product_code"].value_counts().sort_values(ascending=False)
acum = ventas_prod.cumsum() / ventas_prod.sum()
n_prod = len(ventas_prod)
for p in [0.01, 0.10, 0.20, 0.50]:
    print(f"El {p:.0%} de productos más vendidos concentra el "
          f"{100 * acum.iloc[int(p * n_prod) - 1]:.1f} % de las ventas")

orden_asc = np.sort(ventas_prod.to_numpy())
lorenz = np.insert(orden_asc.cumsum() / orden_asc.sum(), 0, 0)
gini = 1 - 2 * np.trapezoid(lorenz, dx=1 / n_prod)
print(f"Productos vendidos en entrenamiento: {n_prod:,} | Coeficiente de Gini: {gini:.3f}")

plt.figure(figsize=(6, 6))
plt.plot(np.linspace(0, 1, n_prod + 1), lorenz, color="steelblue", label="Curva de Lorenz")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Igualdad perfecta")
plt.title("Concentración de las ventas entre productos")
plt.xlabel("Proporción acumulada de productos")
plt.ylabel("Proporción acumulada de ventas")
plt.legend()
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 5.4 Categorías más vendidas
# ------------------------------------------------------------
tr_cat = tr[["product_code"]].join(prod[["index_group_name", "product_group_name"]], on="product_code")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
(tr_cat["index_group_name"].value_counts(normalize=True) * 100).sort_values().plot(
    kind="barh", ax=axes[0], color="steelblue")
axes[0].set_title("Participación por grupo de índice")
axes[0].set_xlabel("Porcentaje de transacciones (%)")
(tr_cat["product_group_name"].value_counts(normalize=True).head(10) * 100).sort_values().plot(
    kind="barh", ax=axes[1], color="darkorange")
axes[1].set_title("Diez grupos de producto más vendidos")
axes[1].set_xlabel("Porcentaje de transacciones (%)")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 5.5 Frecuencia de compra
# ------------------------------------------------------------
dias = tr[["cliente", "fecha"]].drop_duplicates().sort_values(["cliente", "fecha"])
dias["intervalo"] = dias.groupby("cliente")["fecha"].diff().dt.days
intervalos = dias["intervalo"].dropna()
print("\nDías entre fechas de compra consecutivas:")
print(intervalos.describe(percentiles=[.25, .5, .75, .9]).round(1))

plt.figure(figsize=(10, 4))
plt.hist(intervalos.clip(upper=365), bins=73, color="steelblue")
plt.title("Días entre fechas de compra consecutivas (truncado en 365)")
plt.xlabel("Días")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()

**Consistencia de los atributos por producto.** Los atributos de clasificación son prácticamente constantes dentro de cada `product_code`: el porcentaje de productos con más de un valor es inferior al 0,7 % para todos ellos y nulo para la descripción (`detail_desc`). Solo el nombre comercial (`prod_name`) varía en el 4,61 % de los productos, por variantes como «Strap top» y «Strap top (1)». Se toma, por tanto, el primer registro de cada producto como representativo, con un error despreciable en las variables que se usarán en el análisis.

**Evolución temporal.** Las 104 semanas de entrenamiento registran en promedio 30.136 transacciones y 7.658 clientes activos por semana en la muestra. La serie no presenta una tendencia marcada, pero sí picos recurrentes: los más altos ocurren a finales de junio de 2019 y de 2020 (cerca de 55.000 transacciones semanales) y a finales de noviembre de 2019 (cerca de 51.000). Su repetición sugiere temporadas de rebajas, aunque la base no permite verificarlo. Se observa también una caída de la actividad entre marzo y abril de 2020. Los valores extremos de la primera y la última semana (17.000 y 7.642 transacciones) no son caídas reales, sino semanas incompletas en los bordes del período.

**Canal de venta.** El 70,55 % de las transacciones se realiza en línea y el 29,45 % en tienda física. La participación es estable en la mayor parte del período, con dos excepciones. En diciembre de 2019 la tienda física gana peso hasta acercarse al 50 %. Entre finales de marzo y mediados de mayo de 2020, la participación de la tienda cae a cero y todas las ventas ocurren en línea, lo que coincide con el período de cierres por la pandemia de COVID-19 y con la caída de actividad observada en la serie semanal. Esta asociación es una interpretación del calendario, no un hecho registrado en la base.

**Concentración de la demanda.** Las ventas están fuertemente concentradas: el 1 % de los productos más vendidos acumula el 28,0 % de las transacciones, el 10 % acumula el 69,8 % y la mitad de los productos explica el 96,8 %, con un coeficiente de Gini de 0,801. Esta estructura favorece a los recomendadores de popularidad, que acertarán con frecuencia al proponer los productos más vendidos, pero también advierte sobre el riesgo de que los modelos se concentren en una fracción pequeña del catálogo y descuiden la cola larga. Por ello la evaluación incluirá la cobertura del catálogo además de las métricas de precisión.

**Categorías.** La demanda se concentra en ropa femenina: Ladieswear representa cerca del 64 % de las transacciones y Divided cerca del 23 %, mientras que Menswear, Sport y Baby/Children no superan el 6 % cada uno. Por tipo de producto predominan las prendas superiores (cerca del 39 %), seguidas de las inferiores (cerca del 22 %) y las de cuerpo completo (cerca del 11 %).

**Frecuencia de compra.** La mediana del intervalo entre fechas de compra consecutivas es de 22 días y la media de 48, con una distribución muy asimétrica: el 25 % de los intervalos no supera 7 días, pero el 10 % supera 123. La acumulación en 365 días del histograma corresponde al truncamiento de los intervalos mayores. Con una mediana cercana a tres semanas, la mayoría de los clientes no compra en una semana dada, lo que es coherente con que solo cerca del 5 % compre en las semanas evaluadas. La recencia de la última compra será, por tanto, una variable relevante en los perfiles.

### **6. Perfiles de clientes y reducción de dimensión**

#### **6.1 Construcción de los perfiles**

Cada cliente con al menos una compra en el período de entrenamiento se describe mediante un perfil de comportamiento calculado únicamente con su historia previa al inicio de la ventana evaluada. Las variables se agrupan en cuatro bloques:

- **Recencia, frecuencia y valor monetario (RFM)**: días desde la última compra (`recencia`), días desde la primera compra (`antiguedad`), número de fechas de compra (`frecuencia`), número de transacciones, gasto total, número de productos distintos, precio medio por artículo y gasto medio por fecha de compra. Fader, Hardie y Lee (2005) muestran que la recencia, la frecuencia y el valor monetario resumen adecuadamente el comportamiento futuro de compra.
- **Canal y dinámica reciente**: proporción de compras en línea y proporción de transacciones realizadas en las últimas 12 semanas.
- **Preferencias por categoría**: proporción de las compras en cada grupo de índice (Ladieswear, Divided, Menswear, Sport) y en los grupos de producto más vendidos. Como las proporciones de cada bloque suman uno, se omite una categoría por bloque (Baby/Children y Otros) para evitar dependencia lineal exacta.
- **Demografía**: edad del cliente, imputada con la mediana en el 1,2 % de los casos sin registro.

#### **6.2 Transformación y estandarización**

Las variables de conteo y monto presentan fuerte asimetría (sección 3.1), por lo que se transforman con $\log(1 + x)$ antes de estandarizarlas. Todas las variables se estandarizan a media cero y varianza uno, ya que se miden en escalas muy distintas y el ACP es sensible a la escala. El escalador se ajusta con los perfiles del escenario de validación y se reutiliza, sin reajustar, para transformar los perfiles del escenario de prueba.

#### **6.3 Análisis de componentes principales**

Sobre los perfiles estandarizados se aplica el ACP para reducir la dimensión y eliminar la redundancia entre variables correlacionadas, como el número de transacciones y el gasto. El número de componentes se elige con tres criterios: el gráfico de sedimentación (*scree plot*), la proporción de varianza explicada acumulada y el criterio de Kaiser, que retiene las componentes con valor propio mayor que uno. Las cargas de las componentes retenidas permiten interpretar las dimensiones del comportamiento de compra.

In [ ]:
# ============================================================
# 6. PERFILES DE CLIENTES Y REDUCCIÓN DE DIMENSIÓN
# ============================================================

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

GRUPOS_PRODUCTO = ["Garment Upper body", "Garment Lower body", "Garment Full body",
                   "Swimwear", "Underwear", "Accessories", "Shoes"]
GRUPOS_INDICE = ["Ladieswear", "Divided", "Menswear", "Sport"]   # Baby/Children omitido
VARS_LOG = ["recencia", "antiguedad", "frecuencia", "transacciones", "gasto",
            "productos_distintos", "ticket_medio"]

demo = muestra.reset_index(drop=True)[["age"]].assign(cliente=np.arange(len(muestra)))

# ------------------------------------------------------------
# 6.1 Construcción de los perfiles
# ------------------------------------------------------------
def perfiles_clientes(fin_historia):
    """Perfil de comportamiento de cada cliente con historia hasta fin_historia."""
    h = tx_m[tx_m["fecha"] <= fin_historia].join(
        prod[["index_group_name", "product_group_name"]], on="product_code")
    h["online"] = (h["sales_channel_id"] == 2).astype(np.float32)
    h["reciente"] = (h["fecha"] > fin_historia - pd.Timedelta(weeks=12)).astype(np.float32)
    g = h.groupby("cliente")

    p = pd.DataFrame({
        "recencia": (fin_historia - g["fecha"].max()).dt.days,
        "antiguedad": (fin_historia - g["fecha"].min()).dt.days,
        "frecuencia": g["fecha"].nunique(),
        "transacciones": g.size(),
        "gasto": g["price"].sum(),
        "productos_distintos": g["product_code"].nunique(),
        "prop_online": g["online"].mean(),
        "prop_reciente": g["reciente"].mean(),
    })
    p["precio_medio"] = p["gasto"] / p["transacciones"]
    p["ticket_medio"] = p["gasto"] / p["frecuencia"]

    # Proporciones por grupo de índice y grupo de producto
    idx = pd.crosstab(h["cliente"], h["index_group_name"], normalize="index")
    idx = idx.reindex(columns=GRUPOS_INDICE, fill_value=0).add_prefix("idx_")
    grp = pd.crosstab(h["cliente"], h["product_group_name"], normalize="index")
    grp = grp.reindex(columns=GRUPOS_PRODUCTO, fill_value=0).add_prefix("grp_")

    p = p.join(idx).join(grp)
    p = p.join(demo.set_index("cliente")["age"])
    return p

perf_val = perfiles_clientes(corte_val)
edad_mediana = perf_val["age"].median()                 # Mediana del escenario de validación
perf_val["age"] = perf_val["age"].fillna(edad_mediana)

print(f"Clientes con perfil (validación): {len(perf_val):,} | Variables: {perf_val.shape[1]}")
print(perf_val.describe().T.round(3))

# ------------------------------------------------------------
# 6.2 Transformación y estandarización
# ------------------------------------------------------------
def transformar(p):
    """Aplica log(1 + x) a las variables asimétricas."""
    q = p.copy()
    q[VARS_LOG] = np.log1p(q[VARS_LOG])
    return q

X_val = transformar(perf_val)
escalador = StandardScaler().fit(X_val)                 # Ajuste solo en validación
Z_val = escalador.transform(X_val)
variables = X_val.columns.tolist()

# Matriz de correlaciones
corr = pd.DataFrame(Z_val, columns=variables).corr()
plt.figure(figsize=(11, 9))
plt.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(label="Correlación de Pearson")
plt.xticks(range(len(variables)), variables, rotation=90)
plt.yticks(range(len(variables)), variables)
plt.title("Matriz de correlaciones de los perfiles (escala transformada)")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 6.3 Análisis de componentes principales
# ------------------------------------------------------------
pca_full = PCA(random_state=SEED).fit(Z_val)
valores_propios = pca_full.explained_variance_
var_exp = pca_full.explained_variance_ratio_
var_acum = np.cumsum(var_exp)

k_kaiser = int((valores_propios > 1).sum())
k_80 = int(np.argmax(var_acum >= 0.80) + 1)
print(f"\nComponentes con valor propio > 1 (Kaiser): {k_kaiser}")
print(f"Componentes para explicar al menos el 80 % de la varianza: {k_80}")

tabla_vp = pd.DataFrame({
    "componente": [f"CP{i+1}" for i in range(len(var_exp))],
    "valor_propio": valores_propios,
    "varianza_%": 100 * var_exp,
    "acumulada_%": 100 * var_acum,
})
print(tabla_vp.round(3).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
comp = np.arange(1, len(var_exp) + 1)
axes[0].plot(comp, valores_propios, marker="o", color="steelblue")
axes[0].axhline(1, color="gray", linestyle="--", label="Criterio de Kaiser")
axes[0].set_title("Gráfico de sedimentación")
axes[0].set_xlabel("Componente")
axes[0].set_ylabel("Valor propio")
axes[0].legend()
axes[1].bar(comp, 100 * var_exp, color="steelblue", label="Individual")
axes[1].plot(comp, 100 * var_acum, marker="o", color="darkorange", label="Acumulada")
axes[1].axhline(80, color="gray", linestyle="--", label="80 %")
axes[1].set_title("Proporción de varianza explicada")
axes[1].set_xlabel("Componente")
axes[1].set_ylabel("Varianza explicada (%)")
axes[1].legend()
plt.tight_layout()
plt.show()

# Cargas de las componentes que retiene el criterio de Kaiser
cargas = pd.DataFrame(pca_full.components_[:k_kaiser].T, index=variables,
                      columns=[f"CP{i+1}" for i in range(k_kaiser)])
print("\nCargas de las componentes retenidas:")
print(cargas.round(3))

# Guardar perfiles para las secciones siguientes
perf_val.to_parquet(f"{RUTA}/perfiles_validacion.parquet")

**Perfiles construidos.** Se obtienen perfiles para 135.160 clientes con al menos una compra hasta el 8 de septiembre de 2020, descritos por 22 variables. La historia típica es corta y espaciada: la mediana de fechas de compra es 3 y la mediana de la recencia es de 148 días, de modo que más de la mitad de los clientes no compra desde hace casi cinco meses. La mediana de la proporción de transacciones en las últimas 12 semanas es cero, es decir, la mayoría de los clientes no registra actividad reciente. La mediana de la proporción de compras en línea es 0,917, lo que indica que la mayoría de los clientes compra casi exclusivamente por ese canal.

**Número de componentes.** Los criterios no coinciden. El criterio de Kaiser retiene 9 componentes, que explican el 73,57 % de la varianza, mientras que el umbral del 80 % requiere 11 (82,45 %). El gráfico de sedimentación muestra una caída pronunciada solo entre la primera componente (valor propio 4,33) y la segunda (2,11); a partir de ahí los valores propios descienden de forma gradual hasta CP12 y no se observa un codo claro. Las componentes 10 a 12 tienen valores propios entre 0,94 y 1,00, prácticamente sobre el umbral de Kaiser, por lo que la frontera entre retenerlas o no es en buena medida arbitraria. Las últimas siete componentes (CP16 a CP22) explican en conjunto menos del 3 % de la varianza y reflejan la redundancia entre variables construidas como razones de otras, como el precio medio y el gasto medio por fecha.

**Interpretación de las componentes.** Las cargas permiten identificar tres tipos de dimensiones:

- **CP1, intensidad de compra (19,7 %)**: cargas altas y positivas en transacciones (0,471), productos distintos (0,470), gasto (0,449) y frecuencia (0,448), y negativa en recencia (−0,297). Ordena a los clientes desde los ocasionales hasta los compradores frecuentes y recientes.
- **CP2 y CP3, momento de compra y línea de producto (9,6 % y 8,6 %)**: ambas contraponen a los clientes con actividad reciente (prop_reciente 0,430 y 0,422) frente a los antiguos (antigüedad −0,404 y −0,335). CP2 asocia la actividad reciente con Ladieswear (0,423) y trajes de baño (0,300); CP3 la asocia con la línea Divided (0,453) y la contrapone a Ladieswear (−0,446), al precio medio (−0,198) y a la edad (−0,192).
- **CP4, valor por compra y canal (7,9 %)**: precio medio (0,475), gasto por fecha (0,450) y proporción en línea (0,429).
- **CP5 a CP9, especialización por categoría (entre 6,7 % y 4,8 %)**: cada una se asocia con un tipo de prenda o línea específica: prendas superiores (CP5, 0,569), Sport (CP6, 0,535), edad y prendas inferiores frente a trajes de baño (CP7), Menswear, calzado y accesorios (CP8) y ropa interior y calzado frente a trajes de baño (CP9).

**Decisión.** Se retienen las **9 componentes del criterio de Kaiser**. Las componentes 10 y 11, que se agregarían para alcanzar el 80 %, tienen valores propios inferiores a uno y aportan cada una menos que una variable original estandarizada; además, en el agrupamiento, cada dimensión adicional de baja varianza diluye las distancias entre clientes. Como la frontera es estrecha, en la sección 7 se verifica la sensibilidad de la segmentación a esta decisión comparando la partición obtenida con 9 y con 11 componentes.

### **7. Segmentación de clientes**

La segmentación agrupa a los clientes según su comportamiento de compra, representado por los puntajes en las 9 componentes principales retenidas. Se comparan cinco métodos: K-medias, K-medoides, clustering jerárquico de Ward, mezclas gaussianas y DBSCAN. Todos operan sobre el mismo espacio de componentes con distancia euclidiana, lo que hace comparables sus resultados. No se emplea la distancia de Gower, pues tras el ACP el espacio de perfiles es completamente numérico.

#### **7.1 Elección del número de grupos**

El número de grupos se explora en el rango $k = 2, \dots, 15$ con los criterios propios de cada método:

- **K-medias**: método del codo sobre la varianza intra-grupo (*inertia*) y coeficiente de silueta, complementados con los índices de Calinski-Harabasz (mayor es mejor) y Davies-Bouldin (menor es mejor).
- **K-medoides**: como el algoritmo PAM requiere la matriz completa de distancias, de orden $O(n^2)$, no es viable con 135.160 clientes. Se aplica la estrategia **CLARA** (Kaufman y Rousseeuw, 1990): se ejecuta PAM sobre 5 submuestras aleatorias de 5.000 clientes con el algoritmo FasterPAM (Schubert y Rousseeuw, 2021) y se conservan los medoides cuya suma de distancias sobre todos los clientes es mínima.
- **Jerárquico de Ward**: dendrograma y coeficiente de silueta para cortes en $k$ grupos, sobre una submuestra de 10.000 clientes.
- **Mezclas gaussianas**: criterios de información BIC y AIC con matrices de covarianza completas, sobre una submuestra de 30.000 clientes.
- **DBSCAN**: no requiere fijar $k$; su parámetro $\varepsilon$ se elige con la curva de distancias al $k$-ésimo vecino más cercano, con `min_samples` igual a dos veces el número de dimensiones, y el punto de máxima curvatura se identifica con `KneeLocator`.

La silueta se calcula sobre una submuestra de 10.000 clientes, pues su costo también es cuadrático en el número de observaciones.

In [ ]:
# ============================================================
# 7. SEGMENTACIÓN DE CLIENTES
# ============================================================

!pip install -q kmedoids kneed

import time
import kmedoids
from kneed import KneeLocator
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (silhouette_score, calinski_harabasz_score,
                             davies_bouldin_score, pairwise_distances)
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# ------------------------------------------------------------
# 7.1 Elección del número de grupos
# ------------------------------------------------------------
K_PCA = 9
pca = PCA(n_components=K_PCA, random_state=SEED).fit(Z_val)
F_val = pca.transform(Z_val)                       # Puntajes en las componentes retenidas
print(f"Espacio de segmentación: {F_val.shape[0]:,} clientes × {F_val.shape[1]} componentes")

rng7 = np.random.default_rng(SEED)
idx_10k = rng7.choice(len(F_val), 10_000, replace=False)   # Silueta y Ward
idx_30k = rng7.choice(len(F_val), 30_000, replace=False)   # GMM y DBSCAN
rango_k = range(2, 16)

def silueta(F, etiquetas):
    """Silueta sobre una submuestra de 10.000 clientes."""
    return silhouette_score(F, etiquetas, sample_size=10_000, random_state=SEED)

# --- K-medias ---
t0 = time.time()
res_km = []
for k in rango_k:
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(F_val)
    res_km.append({"k": k, "inercia": km.inertia_,
                   "silueta": silueta(F_val, km.labels_),
                   "calinski_harabasz": calinski_harabasz_score(F_val, km.labels_),
                   "davies_bouldin": davies_bouldin_score(F_val, km.labels_)})
res_km = pd.DataFrame(res_km)
print(f"K-medias: {time.time() - t0:.1f} s")

# --- K-medoides con CLARA ---
def clara(F, k, submuestras, rng_seed=SEED):
    """PAM (FasterPAM) en cada submuestra; conserva los medoides de menor costo total."""
    mejor = None
    for idx_s, D_s in submuestras:
        res = kmedoids.fasterpam(D_s, k, random_state=rng_seed)
        medoides = F[idx_s[res.medoids]]
        dist = pairwise_distances(F, medoides)
        costo = dist.min(axis=1).sum()
        if mejor is None or costo < mejor["costo"]:
            mejor = {"costo": costo, "medoides": medoides, "etiquetas": dist.argmin(axis=1)}
    return mejor

t0 = time.time()
submuestras = []
for s in range(5):
    idx_s = rng7.choice(len(F_val), 5_000, replace=False)
    submuestras.append((idx_s, pairwise_distances(F_val[idx_s]).astype(np.float32)))

res_kmd = []
for k in rango_k:
    c = clara(F_val, k, submuestras)
    res_kmd.append({"k": k, "costo": c["costo"], "silueta": silueta(F_val, c["etiquetas"])})
res_kmd = pd.DataFrame(res_kmd)
print(f"K-medoides (CLARA): {time.time() - t0:.1f} s")

# --- Jerárquico de Ward (submuestra de 10.000) ---
t0 = time.time()
F_10k = F_val[idx_10k]
Z_ward = linkage(F_10k, method="ward")
res_ward = pd.DataFrame([{"k": k, "silueta": silhouette_score(F_10k, fcluster(Z_ward, k, criterion="maxclust"))}
                         for k in rango_k])
print(f"Ward: {time.time() - t0:.1f} s")

# --- Mezclas gaussianas (submuestra de 30.000) ---
t0 = time.time()
F_30k = F_val[idx_30k]
res_gmm = []
for k in rango_k:
    gmm = GaussianMixture(n_components=k, covariance_type="full", n_init=2,
                          random_state=SEED).fit(F_30k)
    res_gmm.append({"k": k, "BIC": gmm.bic(F_30k), "AIC": gmm.aic(F_30k)})
res_gmm = pd.DataFrame(res_gmm)
print(f"Mezclas gaussianas: {time.time() - t0:.1f} s")

# --- DBSCAN: curva de distancias al k-ésimo vecino (submuestra de 30.000) ---
min_muestras = 2 * K_PCA
vecinos = NearestNeighbors(n_neighbors=min_muestras).fit(F_30k)
dist_k = np.sort(vecinos.kneighbors(F_30k)[0][:, -1])
rodilla = KneeLocator(np.arange(len(dist_k)), dist_k, curve="convex", direction="increasing")
eps_sugerido = dist_k[rodilla.knee]
print(f"DBSCAN: min_samples = {min_muestras} | eps sugerido = {eps_sugerido:.3f}")

# --- Tablas de resultados ---
tabla_k = (res_km.merge(res_kmd, on="k", suffixes=("_km", "_kmd"))
                 .merge(res_ward.rename(columns={"silueta": "silueta_ward"}), on="k")
                 .merge(res_gmm, on="k"))
print(tabla_k.round(4).to_string(index=False))

# --- Gráficos ---
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes[0, 0].plot(res_km["k"], res_km["inercia"], marker="o", color="steelblue")
axes[0, 0].set_title("K-medias: método del codo")
axes[0, 0].set_ylabel("Varianza intra-grupo")
axes[0, 1].plot(res_kmd["k"], res_kmd["costo"], marker="o", color="darkorange")
axes[0, 1].set_title("K-medoides (CLARA): suma de distancias a los medoides")
axes[0, 1].set_ylabel("Costo")
axes[1, 0].plot(res_km["k"], res_km["silueta"], marker="o", label="K-medias")
axes[1, 0].plot(res_kmd["k"], res_kmd["silueta"], marker="o", label="K-medoides")
axes[1, 0].plot(res_ward["k"], res_ward["silueta"], marker="o", label="Ward")
axes[1, 0].set_title("Coeficiente de silueta")
axes[1, 0].set_ylabel("Silueta")
axes[1, 0].legend()
axes[1, 1].plot(res_gmm["k"], res_gmm["BIC"], marker="o", label="BIC")
axes[1, 1].plot(res_gmm["k"], res_gmm["AIC"], marker="o", label="AIC")
axes[1, 1].set_title("Mezclas gaussianas: criterios de información")
axes[1, 1].set_ylabel("Valor del criterio")
axes[1, 1].legend()
for ax in axes.ravel():
    ax.set_xlabel("Número de grupos (k)")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
dendrogram(Z_ward, truncate_mode="lastp", p=30, leaf_rotation=90, ax=axes[0], color_threshold=None)
axes[0].set_title("Dendrograma de Ward (submuestra de 10.000, últimas 30 fusiones)")
axes[0].set_xlabel("Grupos")
axes[0].set_ylabel("Distancia")
axes[1].plot(dist_k, color="steelblue")
axes[1].axhline(eps_sugerido, color="gray", linestyle="--", label=f"eps sugerido = {eps_sugerido:.3f}")
axes[1].set_title(f"Distancia al vecino {min_muestras} (submuestra de 30.000)")
axes[1].set_xlabel("Clientes ordenados")
axes[1].set_ylabel("Distancia")
axes[1].legend()
plt.tight_layout()
plt.show()

**Tiempo de cómputo.** La exploración completa tardó cerca de 7,5 minutos. Las mezclas gaussianas (176,9 s sobre 30.000 clientes) y K-medoides con CLARA (151,2 s) fueron los métodos más costosos, seguidos de K-medias (85,7 s sobre los 135.160 clientes) y Ward (32,0 s sobre 10.000).

**Método del codo.** Tanto la varianza intra-grupo de K-medias como el costo de K-medoides decrecen de forma suave y continua entre $k = 2$ y $k = 15$, sin un quiebre que identifique un número de grupos. El codo depende de un juicio subjetivo y en este caso no ofrece una recomendación clara.

**Coeficiente de silueta.** Los valores son bajos para todos los métodos y todos los valores de $k$: entre 0,11 y 0,21 para K-medias, entre 0,11 y 0,17 para K-medoides y entre 0,09 y 0,15 para Ward. Esto indica que los clientes no forman grupos bien separados en el espacio de comportamiento, sino un continuo en el que cualquier partición separa regiones contiguas. En K-medias la silueta alcanza un máximo local en $k = 3$ (0,192) y el máximo global en $k = 7$ (0,206), seguido de una caída abrupta en $k = 8$ (0,158). En Ward el máximo se alcanza en $k = 3$ (0,145), y en K-medoides el valor en $k = 3$ (0,162) es cercano a su máximo (0,167 en $k = 11$).

**Índices complementarios.** El índice de Calinski-Harabasz decrece monótonamente con $k$, comportamiento habitual cuando no hay grupos bien definidos. El de Davies-Bouldin alcanza en $k = 3$ su valor más bajo del tramo $k = 2$ a $k = 9$ (1,676), aunque disminuye de nuevo para valores grandes de $k$.

**Dendrograma.** La mayor distancia de fusión separa una rama de cerca de una cuarta parte de la submuestra del resto, y la segunda divide el bloque mayor en dos, con una separación clara frente a las fusiones siguientes. La estructura jerárquica respalda, por tanto, una partición en dos o tres grupos.

**Mezclas gaussianas.** El BIC y el AIC disminuyen de forma sostenida en todo el rango, con un descenso marcado entre $k = 7$ y $k = 8$, y no alcanzan un mínimo. Con matrices de covarianza completas, las componentes adicionales modelan la forma no gaussiana de la distribución (proporciones con alta concentración en cero y variables asimétricas) más que grupos diferenciados, de modo que los criterios de información no identifican un número de segmentos interpretable.

**DBSCAN.** El punto de máxima curvatura de la distancia al vecino 18 se ubica en $\varepsilon = 3{,}074$, en el extremo de la curva: para casi todos los clientes esa distancia está entre 1 y 2, y solo se dispara en el último 1 %. Un valor de $\varepsilon$ en esa zona agrupa a casi todos los clientes en un único grupo denso y deja como ruido solo a los más atípicos, lo que anticipa que DBSCAN funcionará como detector de anomalías más que como método de segmentación.

**Decisión.** Se adopta $k = 3$ para los métodos que requieren fijar el número de grupos. Es el único valor respaldado de forma simultánea por la silueta de K-medias (máximo local), la de Ward (máximo global), el índice de Davies-Bouldin y el dendrograma. Dado que la silueta de K-medias alcanza su máximo en $k = 7$, esta alternativa se evaluará en la sección 8 según su desempeño como base de recomendación en la semana de validación.

#### **7.2 Ajuste y comparación de los modelos**

Con $k = 3$ se ajustan los modelos finales: K-medias y K-medoides (CLARA) sobre los 135.160 clientes, Ward sobre la submuestra de 10.000 y mezclas gaussianas sobre la submuestra de 30.000, extendiendo su asignación a todos los clientes con la regla de máxima probabilidad posterior. DBSCAN se aplica sobre la submuestra de 10.000 con una grilla de valores de $\varepsilon$ (percentiles 50, 75 y 90 de la distancia al vecino 18 y el punto de máxima curvatura), para evaluar si emerge una estructura de grupos para alguno de ellos.

Las particiones se comparan mediante tres criterios:

- **Calidad interna**: tamaño de los grupos, silueta, Calinski-Harabasz y Davies-Bouldin.
- **Acuerdo entre métodos**: índice de Rand ajustado (ARI) entre cada par de particiones sobre la submuestra común de 10.000 clientes.
- **Robustez de K-medias**: acuerdo con las particiones obtenidas con cinco semillas distintas (estabilidad frente a la inicialización) y con 11 componentes principales en lugar de 9 (sensibilidad a la decisión de la sección 6).

Finalmente, los segmentos de K-medias y K-medoides se describen con las medianas y medias de las variables originales del perfil, para interpretarlos en términos de comportamiento de compra.

In [ ]:
# ============================================================
# 7.2 AJUSTE Y COMPARACIÓN DE LOS MODELOS
# ============================================================

from sklearn.metrics import adjusted_rand_score

K_SEG = 3

# --- Modelos finales ---
km_final = KMeans(n_clusters=K_SEG, n_init=10, random_state=SEED).fit(F_val)
lab_km = km_final.labels_

clara_final = clara(F_val, K_SEG, submuestras)
lab_kmd = clara_final["etiquetas"]

lab_ward_10k = fcluster(Z_ward, K_SEG, criterion="maxclust") - 1

gmm_final = GaussianMixture(n_components=K_SEG, covariance_type="full", n_init=2,
                            random_state=SEED).fit(F_30k)
lab_gmm = gmm_final.predict(F_val)

# --- DBSCAN con grilla de eps (submuestra de 10.000) ---
dist_k10 = np.sort(NearestNeighbors(n_neighbors=min_muestras).fit(F_10k).kneighbors(F_10k)[0][:, -1])
rodilla10 = KneeLocator(np.arange(len(dist_k10)), dist_k10, curve="convex", direction="increasing")
grilla_eps = {"P50": np.percentile(dist_k10, 50), "P75": np.percentile(dist_k10, 75),
              "P90": np.percentile(dist_k10, 90), "Rodilla": dist_k10[rodilla10.knee]}

res_db, labs_db = [], {}
for nombre, eps in grilla_eps.items():
    lab = DBSCAN(eps=eps, min_samples=min_muestras).fit_predict(F_10k)
    labs_db[nombre] = lab
    grupos = sorted(set(lab) - {-1})
    tam = pd.Series(lab[lab != -1]).value_counts(normalize=True) * 100
    res_db.append({"eps": nombre, "valor_eps": eps, "grupos": len(grupos),
                   "ruido_%": 100 * (lab == -1).mean(),
                   "grupo_mayor_%": tam.max() if len(tam) else np.nan})
print("DBSCAN (submuestra de 10.000):")
print(pd.DataFrame(res_db).round(3).to_string(index=False))

# --- Calidad interna ---
def metricas(F, lab, nombre):
    """Tamaños, silueta, Calinski-Harabasz y Davies-Bouldin de una partición."""
    mask = lab != -1
    tam = pd.Series(lab[mask]).value_counts(normalize=True).sort_index() * 100
    validos = len(tam) >= 2
    return {"método": nombre, "clientes": len(lab), "grupos": len(tam),
            "tamaños_%": " / ".join(f"{t:.1f}" for t in tam.values),
            "silueta": silhouette_score(F[mask], lab[mask], sample_size=min(10_000, mask.sum()),
                                        random_state=SEED) if validos else np.nan,
            "calinski_harabasz": calinski_harabasz_score(F[mask], lab[mask]) if validos else np.nan,
            "davies_bouldin": davies_bouldin_score(F[mask], lab[mask]) if validos else np.nan}

tabla_seg = pd.DataFrame([
    metricas(F_val, lab_km, "K-medias"),
    metricas(F_val, lab_kmd, "K-medoides (CLARA)"),
    metricas(F_10k, lab_ward_10k, "Ward (10.000)"),
    metricas(F_val, lab_gmm, "Mezclas gaussianas"),
    metricas(F_10k, labs_db["Rodilla"], "DBSCAN (10.000, rodilla)"),
])
print("\nCalidad interna de las particiones:")
print(tabla_seg.round(4).to_string(index=False))

# --- Acuerdo entre métodos (ARI sobre la submuestra común de 10.000) ---
particiones = {"K-medias": lab_km[idx_10k], "K-medoides": lab_kmd[idx_10k],
               "Ward": lab_ward_10k, "GMM": lab_gmm[idx_10k]}
ari = pd.DataFrame({a: {b: adjusted_rand_score(pa, pb) for b, pb in particiones.items()}
                    for a, pa in particiones.items()})
print("\nARI entre métodos (submuestra de 10.000):")
print(ari.round(3))

# --- Robustez de K-medias ---
ari_semillas = [adjusted_rand_score(lab_km, KMeans(n_clusters=K_SEG, n_init=10, random_state=s)
                                    .fit_predict(F_val)) for s in [1, 2, 3, 4, 5]]
F_11 = PCA(n_components=11, random_state=SEED).fit_transform(Z_val)
ari_11 = adjusted_rand_score(lab_km, KMeans(n_clusters=K_SEG, n_init=10, random_state=SEED)
                             .fit_predict(F_11))
print(f"\nK-medias, ARI frente a otras semillas: media {np.mean(ari_semillas):.3f} "
      f"(mín. {np.min(ari_semillas):.3f})")
print(f"K-medias, ARI entre 9 y 11 componentes: {ari_11:.3f}")

# --- Visualización en el plano CP1–CP2 (submuestra de 10.000) ---
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
paneles = list(particiones.items()) + [("DBSCAN (rodilla)", labs_db["Rodilla"]),
                                       ("DBSCAN (P75)", labs_db["P75"])]
for ax, (nombre, lab) in zip(axes.ravel(), paneles):
    ax.scatter(F_10k[:, 0], F_10k[:, 1], c=np.where(lab == -1, -1, lab),
               cmap="tab10", s=4, alpha=0.6)
    if (lab == -1).any():
        ax.scatter(F_10k[lab == -1, 0], F_10k[lab == -1, 1], c="black", s=6, label="Ruido")
        ax.legend()
    ax.set_title(nombre)
    ax.set_xlabel("CP1: intensidad de compra")
    ax.set_ylabel("CP2: actividad reciente")
plt.tight_layout()
plt.show()

# --- Perfil de los segmentos ---
def perfil_segmentos(lab, nombre):
    """Medianas de RFM y medias de proporciones por segmento."""
    p = perf_val.assign(segmento=lab)
    medianas = p.groupby("segmento")[["recencia", "antiguedad", "frecuencia", "transacciones",
                                      "gasto", "precio_medio", "age"]].median()
    medias = p.groupby("segmento")[["prop_online", "prop_reciente"]
                                   + [c for c in p.columns if c.startswith(("idx_", "grp_"))]].mean()
    tabla = medianas.join(medias)
    tabla.insert(0, "clientes_%", 100 * p["segmento"].value_counts(normalize=True).sort_index())
    print(f"\nPerfil de los segmentos: {nombre} (medianas de RFM y edad; medias de proporciones)")
    print(tabla.T.round(3))
    return tabla

perfil_km  = perfil_segmentos(lab_km, "K-medias")
perfil_kmd = perfil_segmentos(lab_kmd, "K-medoides (CLARA)")

**DBSCAN.** Ningún valor de $\varepsilon$ produce una segmentación útil. Con el percentil 50 se obtienen 9 grupos, pero el 25,8 % de los clientes queda como ruido y el grupo mayor reúne el 91,3 % del resto. Al aumentar $\varepsilon$ el ruido disminuye y los grupos pequeños se absorben en el principal, hasta que en el punto de máxima curvatura queda un único grupo con el 99,7 % de los clientes y un 0,3 % de atípicos, ubicados en los extremos del plano de las componentes. Los clientes forman, por tanto, una nube de densidad continua sin regiones densas separadas por zonas vacías, que es el supuesto bajo el cual DBSCAN identifica grupos. En este conjunto de datos el método resulta útil como detector de clientes atípicos, no como técnica de segmentación.

**Calidad interna.** K-medias obtiene la mejor silueta (0,192) y el mayor índice de Calinski-Harabasz (25.230), con grupos de tamaño desigual (36,3 %, 55,5 % y 8,1 %). Ward, sobre la submuestra, logra el menor Davies-Bouldin (1,614) con una partición aún más desbalanceada (25,8 %, 5,1 % y 69,2 %). K-medoides produce los grupos más equilibrados (28,0 %, 33,1 % y 38,9 %), a costa de una silueta menor (0,162) y un Davies-Bouldin mayor (2,280). Las mezclas gaussianas obtienen los peores valores en los tres índices (silueta de 0,098 y Davies-Bouldin de 2,888), coherente con lo observado en la sección 7.1: sus componentes modelan la forma de la distribución y se superponen en el plano, en lugar de separar regiones.

**Acuerdo entre métodos.** K-medias coincide moderadamente con Ward (ARI de 0,515) y con K-medoides (0,483), mientras que las mezclas gaussianas apenas coinciden con los demás (ARI entre 0,038 y 0,150). La mayor similitud entre K-medias y Ward es la esperada por teoría: ambos minimizan la varianza intra-grupo, uno de forma iterativa y el otro de forma aglomerativa.

**Robustez de K-medias.** La partición es idéntica con cinco semillas distintas (ARI de 1,000) y prácticamente igual con 11 componentes en lugar de 9 (ARI de 0,996). La solución es estable frente a la inicialización, y la decisión sobre el número de componentes de la sección 6 no la afecta.

**Segmentos de K-medias.** Los tres segmentos se diferencian por la intensidad y el momento de la compra, no por las preferencias:

- **Segmento 0, clientes frecuentes (36,3 %)**: mediana de 11 fechas de compra, 37 transacciones y 43 días desde la última compra, con 673 días de antigüedad.
- **Segmento 1, clientes ocasionales inactivos (55,5 %)**: una sola fecha de compra, 4 transacciones y 335 días desde la última compra; solo el 2,2 % de sus transacciones ocurrió en las últimas 12 semanas.
- **Segmento 2, clientes nuevos (8,1 %)**: 71 días de antigüedad y 86,4 % de sus transacciones en las últimas 12 semanas. Son algo más jóvenes (mediana de 27 años) y compran más trajes de baño y prendas de cuerpo completo.

Fuera de estas diferencias, las proporciones por categoría son casi idénticas entre segmentos (por ejemplo, Ladieswear entre 0,60 y 0,67), de modo que K-medias separa a los clientes según las dos primeras componentes y no según sus gustos.

**Segmentos de K-medoides.** K-medoides identifica también un grupo de clientes frecuentes (33,1 %, con mediana de 40 transacciones y 38 días de recencia), pero divide a los ocasionales según su **línea de producto**:

- **Segmento 0, ocasionales de Divided y Menswear (28,0 %)**: Ladieswear representa solo el 27,1 % de sus compras, frente al 46,0 % de Divided y el 11,8 % de Menswear; predominan las prendas superiores e inferiores y casi no compran trajes de baño ni ropa interior.
- **Segmento 2, ocasionales de Ladieswear (38,9 %)**: el 85,3 % de sus compras es Ladieswear, con mayor peso de trajes de baño (13,8 %), ropa interior (12,4 %) y accesorios.

**Interpretación.** La diferencia entre ambos métodos puede deberse a su función objetivo. K-medias minimiza distancias al cuadrado, lo que da gran peso a las componentes de mayor varianza (CP1, con valor propio 4,33) y a los valores extremos de la actividad reciente. K-medoides minimiza distancias sin elevar al cuadrado, lo que reduce ese predominio y permite que las componentes de preferencia por categoría influyan más en la partición. Esta explicación es una hipótesis que no se ha verificado formalmente.

**Conclusión.** K-medias ofrece la partición más compacta y estable, pero sus segmentos describen la actividad del cliente y no sus preferencias. K-medoides sacrifica algo de calidad interna a cambio de segmentos diferenciados por línea de producto, potencialmente más útiles para recomendar. Las mezclas gaussianas y DBSCAN no producen segmentos útiles en este conjunto de datos. Por ello, en la sección 8 se evalúan como bases de recomendación las particiones de K-medias ($k = 3$ y $k = 7$) y de K-medoides ($k = 3$), y la semana de validación permite determinar cuál resulta más útil para recomendar.

### **8. Sistemas de recomendación individuales**

En esta sección se implementan los recomendadores individuales y se evalúan en el escenario de validación: entrenamiento con la historia hasta el 8 de septiembre de 2020 y evaluación con las compras de la semana del 9 al 15 de septiembre. Las configuraciones que mejor se desempeñen en validación se reentrenan y evalúan en la semana de prueba en la sección 10.

#### **8.1 Marco de evaluación**

A cada uno de los 7.209 compradores de la semana de validación se le asigna una lista ordenada de 12 productos del conjunto de candidatos. Si un recomendador genera menos de 12 productos para un cliente, la lista se completa con los productos más populares, de modo que todos los modelos se comparan con listas de igual longitud. Las métricas son:

- **Precisión@12**: proporción de los 12 productos recomendados que el cliente compró.
- **Recall@12**: proporción de los productos comprados por el cliente que aparecen en la lista.
- **MAP@12**: promedio sobre los clientes de la precisión promedio, $AP@12 = \frac{1}{\min(m, 12)} \sum_{i=1}^{12} P(i)\,\text{rel}(i)$, donde $m$ es el número de productos comprados, $P(i)$ la precisión en la posición $i$ y $\text{rel}(i)$ vale uno si el producto en la posición $i$ fue comprado. A diferencia de la precisión y el recall, premia que los aciertos aparezcan en las primeras posiciones.
- **Cobertura**: proporción del conjunto de candidatos que aparece en al menos una lista. Mide la diversidad del recomendador y su exposición al sesgo de popularidad identificado en la sección 5.

#### **8.2 Recomendadores de referencia**

- **Popularidad global**: recomienda a todos los clientes los 12 productos más vendidos en las últimas $W$ semanas, en el espíritu de un recomendador simple. La ventana $W \in \{1, 2, 4, 8, 12\}$ se elige en validación; dada la rotación del catálogo, se espera que las ventanas cortas funcionen mejor.
- **Historial del cliente**: recomienda los productos que el propio cliente compró, ordenados por la fecha de su última compra y, en caso de empate, por el número de veces que los compró. Captura la recompra, que según la sección 4 representa cerca del 8 % de las compras evaluadas.

#### **8.3 Popularidad por segmento**

Cada cliente recibe los productos más vendidos en las últimas $W$ semanas entre los clientes de su mismo segmento, con la ventana elegida para la popularidad global. Se comparan cuatro criterios de agrupación: los segmentos de K-medias con $k = 3$ y $k = 7$, los de K-medoides con $k = 3$ y, como referencia demográfica, los grupos de edad, en la línea del recomendador demográfico clásico (por género y ocupación). Los clientes sin compras previas no tienen perfil y no pueden asignarse a un segmento de comportamiento; para ellos se usa la popularidad de su grupo de edad.

In [ ]:
# ============================================================
# 8. SISTEMAS DE RECOMENDACIÓN INDIVIDUALES
# ============================================================

K_REC = 12

# ------------------------------------------------------------
# 8.1 Marco de evaluación (escenario de validación)
# ------------------------------------------------------------
hist_val = tx_m[tx_m["fecha"] <= corte_val]
obj_val  = tx_m[(tx_m["fecha"] > corte_val) & (tx_m["fecha"] <= corte_test)]
cand_val = escenarios["Validación"]["candidatos"]

verdad_val = obj_val.groupby("cliente")["product_code"].apply(set).to_dict()
clientes_eval = np.array(sorted(verdad_val))
print(f"Compradores evaluados: {len(clientes_eval):,} | Candidatos: {len(cand_val):,}")

def ap_k(rec, real, k=K_REC):
    """Precisión promedio en k."""
    aciertos, suma = 0, 0.0
    for i, p in enumerate(rec[:k]):
        if p in real:
            aciertos += 1
            suma += aciertos / (i + 1)
    return suma / min(len(real), k)

def evaluar(recs, verdad, candidatos, nombre, k=K_REC):
    """MAP@k, Precisión@k, Recall@k y cobertura; devuelve también el AP por cliente."""
    ap, prec, rec_, recomendados = [], [], [], set()
    for c, real in verdad.items():
        r = list(recs.get(c, []))[:k]
        recomendados.update(r)
        h = len(set(r) & real)
        ap.append(ap_k(r, real, k))
        prec.append(h / k)
        rec_.append(h / len(real))
    fila = {"modelo": nombre, "MAP@12": np.mean(ap), "Precisión@12": np.mean(prec),
            "Recall@12": np.mean(rec_), "cobertura_%": 100 * len(recomendados) / len(candidatos)}
    return fila, np.array(ap)

def completar(lista, relleno, k=K_REC):
    """Completa una lista hasta k productos con el relleno, sin repetir."""
    vistos = set(lista)
    return list(lista) + [p for p in relleno if p not in vistos][: k - len(lista)]

resultados_val, ap_val = [], {}
def registrar(recs, nombre):
    fila, ap = evaluar(recs, verdad_val, cand_val, nombre)
    resultados_val.append(fila)
    ap_val[nombre] = ap
    return fila

# ------------------------------------------------------------
# 8.2 Recomendadores de referencia
# ------------------------------------------------------------
def populares(hist, fin_historia, semanas, candidatos, n=K_REC):
    """Productos candidatos más vendidos en las últimas 'semanas'."""
    h = hist[(hist["fecha"] > fin_historia - pd.Timedelta(weeks=semanas))
             & hist["product_code"].isin(candidatos)]
    return h["product_code"].value_counts().index[:n].tolist()

# Popularidad global con distintas ventanas
for W in [1, 2, 4, 8, 12]:
    top = populares(hist_val, corte_val, W, cand_val)
    print(registrar({c: top for c in clientes_eval}, f"Popularidad global ({W} sem.)"))

mejor_pop = max([r for r in resultados_val if r["modelo"].startswith("Popularidad global")],
                key=lambda r: r["MAP@12"])
W_POP = int(mejor_pop["modelo"].split("(")[1].split(" ")[0])
relleno_val = populares(hist_val, corte_val, W_POP, cand_val, n=200)
print(f"\nVentana elegida para la popularidad: {W_POP} semana(s)")

# Historial del cliente: lo comprado más recientemente, luego lo más frecuente
h = hist_val[hist_val["cliente"].isin(clientes_eval) & hist_val["product_code"].isin(cand_val)]
hist_cli = (h.groupby(["cliente", "product_code"])
             .agg(ultima=("fecha", "max"), veces=("fecha", "size"))
             .reset_index()
             .sort_values(["cliente", "ultima", "veces"], ascending=[True, False, False]))
listas_hist = hist_cli.groupby("cliente")["product_code"].apply(lambda s: s.head(K_REC).tolist()).to_dict()
recs_hist = {c: completar(listas_hist.get(c, []), relleno_val) for c in clientes_eval}
print(registrar(recs_hist, "Historial del cliente"))

# ------------------------------------------------------------
# 8.3 Popularidad por segmento
# ------------------------------------------------------------
edad_cliente = pd.Series(muestra.reset_index(drop=True)["estrato_edad"].astype(str).to_numpy(),
                         index=np.arange(len(muestra)))

km7 = KMeans(n_clusters=7, n_init=10, random_state=SEED).fit(F_val)
particiones_seg = {
    "K-medias (k=3)":   pd.Series(lab_km, index=perf_val.index),
    "K-medias (k=7)":   pd.Series(km7.labels_, index=perf_val.index),
    "K-medoides (k=3)": pd.Series(lab_kmd, index=perf_val.index),
}

def grupos_con_respaldo(seg):
    """Segmento de comportamiento; clientes sin perfil reciben su grupo de edad."""
    g = edad_cliente.map(lambda e: f"edad {e}")
    g.loc[seg.index] = "seg " + seg.astype(str)
    return g

def recs_por_grupo(grupo_cliente, hist, fin_historia, semanas, candidatos, relleno, clientes):
    """Productos más vendidos en las últimas 'semanas' dentro del grupo de cada cliente."""
    h = hist[(hist["fecha"] > fin_historia - pd.Timedelta(weeks=semanas))
             & hist["product_code"].isin(candidatos)]
    h = h.assign(grupo=h["cliente"].map(grupo_cliente))
    top = h.groupby("grupo")["product_code"].value_counts().groupby(level=0).head(K_REC)
    listas = top.reset_index().groupby("grupo")["product_code"].apply(list).to_dict()
    # Respaldo de edad para clientes sin perfil
    h_edad = h.assign(grupo_edad=h["cliente"].map(edad_cliente))
    top_e = h_edad.groupby("grupo_edad")["product_code"].value_counts().groupby(level=0).head(K_REC)
    listas_edad = top_e.reset_index().groupby("grupo_edad")["product_code"].apply(list).to_dict()
    recs = {}
    for c in clientes:
        g = grupo_cliente.get(c)
        base = listas.get(g) if g in listas else listas_edad.get(edad_cliente[c], [])
        recs[c] = completar(base, relleno)
    return recs

for nombre, seg in particiones_seg.items():
    recs = recs_por_grupo(grupos_con_respaldo(seg), hist_val, corte_val, W_POP,
                          cand_val, relleno_val, clientes_eval)
    print(registrar(recs, f"Segmento: {nombre}"))

recs_edad = recs_por_grupo(edad_cliente.map(lambda e: f"edad {e}"), hist_val, corte_val, W_POP,
                           cand_val, relleno_val, clientes_eval)
print(registrar(recs_edad, "Segmento: grupos de edad"))

# Tamaño de los segmentos de K-medias (k=7) para interpretar el resultado
print("\nTamaño de los segmentos de K-medias (k=7), %:")
print((pd.Series(km7.labels_).value_counts(normalize=True).sort_index() * 100).round(1))

# Tabla de resultados en validación
pd.DataFrame(resultados_val).round(5)

**Popularidad global.** El desempeño de la popularidad disminuye de forma monótona con la longitud de la ventana: el MAP@12 pasa de 0,0198 con la última semana a 0,0083 con las últimas 12, una caída del 58 %. La demanda de moda cambia con rapidez y los productos más vendidos hace dos o tres meses dejan de serlo, lo que confirma lo observado en la rotación del catálogo (sección 4). Se adopta una ventana de **una semana** para todos los recomendadores basados en popularidad. Su cobertura es de apenas 0,06 % del catálogo, pues recomienda los mismos 12 productos a todos los clientes.

**Historial del cliente.** Recomendar lo que el propio cliente compró más recientemente alcanza un MAP@12 de 0,0454, 2,3 veces el de la mejor popularidad global, y un Recall@12 de 0,0890. El Recall supera la proporción de recompras de la sección 4 (8,12 %) porque las listas se completan con los productos populares, que aportan aciertos adicionales. El MAP es alto en relación con la precisión porque los aciertos por recompra aparecen en las primeras posiciones de la lista, ordenadas por recencia. Es además el recomendador más diverso (53,8 % del catálogo), pues cada cliente recibe una lista propia.

**Popularidad por segmento.** El aporte de la segmentación es pequeño y depende del método. Los segmentos de K-medoides obtienen el mejor MAP@12 (0,0204), un 3,0 % superior a la popularidad global, seguidos de los grupos de edad (0,0200, +1,2 %). En cambio, los segmentos de K-medias empeoran la recomendación: con $k = 7$ el MAP@12 es 4,9 % inferior al global (0,0188) y con $k = 3$ es 14,4 % inferior (0,0169).

**Interpretación.** Este resultado es coherente con los perfiles de la sección 7. Los segmentos de K-medias se distinguen por la intensidad y el momento de la compra, no por los gustos, de modo que calcular la popularidad dentro de cada uno no aporta información sobre qué productos preferirá el cliente; además, con una ventana de una semana, los segmentos de clientes inactivos registran pocas compras y sus listas de populares se estiman con mucho ruido. Los segmentos de K-medoides se diferencian por línea de producto (Ladieswear frente a Divided y Menswear), que es justamente la información que permite ajustar la popularidad a cada cliente. Esta explicación es una hipótesis compatible con los perfiles, no una relación verificada.

**Decisión.** Para el modelo híbrido se adoptan la popularidad de la última semana como señal general, el historial del cliente como señal individual y los segmentos de K-medoides con $k = 3$ como señal de grupo, que además resuelve el caso de $k = 3$ frente a $k = 7$ planteado en la sección 7.1. Como las diferencias entre los recomendadores por segmento son pequeñas, su significancia se evalúa a continuación mediante intervalos de confianza *bootstrap* pareados.

#### **8.4 Significancia de las diferencias**

Como todos los recomendadores se evalúan sobre los mismos 7.209 compradores, las diferencias de MAP@12 entre dos modelos se analizan de forma pareada: para cada cliente se calcula la diferencia entre sus precisiones promedio y se obtiene un intervalo de confianza del 95 % para la diferencia media mediante 2.000 remuestreos *bootstrap* de clientes. Si el intervalo no contiene el cero, existe evidencia estadística de que los dos recomendadores difieren en su desempeño medio.

#### **8.5 Reglas de asociación (Apriori)**

Siguiendo el análisis de canasta de compras, se define una **canasta** como el conjunto de productos que un cliente compra en una misma fecha. Con las canastas de las últimas 8 semanas de entrenamiento se obtienen, mediante el algoritmo Apriori (`apyori`), reglas de asociación entre pares de productos con soporte, confianza y *lift*. La longitud de las reglas se limita a dos productos para mantener un tiempo de cómputo razonable sobre un catálogo de más de 20.000 productos. Para recomendar, cada producto candidato recibe como puntaje la suma de las confianzas de las reglas cuyo antecedente está entre los 20 productos que el cliente compró más recientemente.

#### **8.6 Filtrado colaborativo basado en usuarios**

Se construye la matriz binaria $R$ de clientes × productos candidatos, con $r_{ij} = 1$ si el cliente $i$ compró el producto $j$ en el período de entrenamiento. La similitud entre clientes se mide con el coseno; sin embargo, la matriz de similitudes completa tendría cerca de $137.198^2 \approx 1{,}9 \times 10^{10}$ entradas, por lo que se calcula por lotes, solo para los compradores evaluados, conservando sus $K$ vecinos más similares. El puntaje de cada producto es la suma de las similitudes de los vecinos que lo compraron. El número de vecinos $K \in \{20, 50, 100\}$ se elige en validación.

#### **8.7 Factorización de matrices (SVD)**

La matriz $R$ se factoriza mediante la descomposición en valores singulares truncada (`svds`), $R \approx U_d \Sigma_d V_d^{\top}$, y el puntaje de cada cliente para cada producto se obtiene de la reconstrucción de rango $d$. A diferencia de la implementación habitual, la matriz **no se centra** restando la media de cada cliente: el centrado convertiría la matriz dispersa en una densa de cerca de 2.800 millones de entradas, inviable en memoria, y en datos implícitos las celdas vacías no representan una calificación media sino la ausencia de compra. La dimensión latente $d \in \{32, 64, 128\}$ se elige en validación.

In [ ]:
# ============================================================
# 8.4 – 8.7 SIGNIFICANCIA, APRIORI, FILTRADO COLABORATIVO Y SVD
# ============================================================

!pip install -q apyori

from apyori import apriori
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
from sklearn.preprocessing import normalize

# ------------------------------------------------------------
# 8.4 Significancia de las diferencias (bootstrap pareado)
# ------------------------------------------------------------
def comparar_pareado(a, b, B=2000, seed=SEED):
    """IC del 95 % bootstrap para la diferencia media de AP@12 entre dos modelos."""
    d = ap_val[a] - ap_val[b]
    rng_b = np.random.default_rng(seed)
    boot = np.array([d[rng_b.integers(0, len(d), len(d))].mean() for _ in range(B)])
    li, ls = np.percentile(boot, [2.5, 97.5])
    return {"comparación": f"{a} − {b}", "diferencia_MAP": d.mean(),
            "IC95_inf": li, "IC95_sup": ls, "significativa": not (li <= 0 <= ls)}

REF = "Popularidad global (1 sem.)"
comparaciones = [comparar_pareado(m, REF) for m in
                 ["Historial del cliente", "Segmento: K-medoides (k=3)",
                  "Segmento: grupos de edad", "Segmento: K-medias (k=7)", "Segmento: K-medias (k=3)"]]
print(pd.DataFrame(comparaciones).round(5).to_string(index=False))

# ------------------------------------------------------------
# 8.5 Reglas de asociación (Apriori)
# ------------------------------------------------------------
SEM_CANASTA, MIN_SOPORTE = 8, 0.0005
h = hist_val[(hist_val["fecha"] > corte_val - pd.Timedelta(weeks=SEM_CANASTA))
             & hist_val["product_code"].isin(cand_val)]
canastas = h.groupby(["cliente", "fecha"])["product_code"].apply(lambda s: list(set(s))).tolist()
print(f"\nCanastas: {len(canastas):,} | Con 2 o más productos: {sum(len(c) >= 2 for c in canastas):,}")

t0 = time.time()
reglas = list(apriori(canastas, min_support=MIN_SOPORTE, min_confidence=0.01,
                      min_lift=1.0, max_length=2))
filas = []
for r in reglas:
    for o in r.ordered_statistics:
        if len(o.items_base) == 1 and len(o.items_add) == 1:
            filas.append({"antecedente": next(iter(o.items_base)), "consecuente": next(iter(o.items_add)),
                          "soporte": r.support, "confianza": o.confidence, "lift": o.lift})
reglas_df = pd.DataFrame(filas)
print(f"Apriori: {time.time() - t0:.1f} s | Reglas obtenidas: {len(reglas_df):,}")

top_reglas = reglas_df.sort_values("lift", ascending=False).head(10).copy()
top_reglas["antecedente_nombre"] = top_reglas["antecedente"].map(prod["prod_name"])
top_reglas["consecuente_nombre"] = top_reglas["consecuente"].map(prod["prod_name"])
print("\nDiez reglas con mayor lift:")
print(top_reglas[["antecedente_nombre", "consecuente_nombre", "soporte", "confianza", "lift"]]
      .round(4).to_string(index=False))

# Recomendación: suma de confianzas de las reglas activadas por las compras recientes
reglas_por_ant = reglas_df.groupby("antecedente")[["consecuente", "confianza"]].apply(
    lambda g: list(zip(g["consecuente"], g["confianza"]))).to_dict()
recientes = hist_cli.groupby("cliente")["product_code"].apply(lambda s: s.head(20).tolist()).to_dict()

recs_apriori, con_reglas = {}, 0
for c in clientes_eval:
    puntaje = {}
    for a in recientes.get(c, []):
        for cons, conf in reglas_por_ant.get(a, []):
            puntaje[cons] = puntaje.get(cons, 0) + conf
    if puntaje:
        con_reglas += 1
    lista = [p for p, _ in sorted(puntaje.items(), key=lambda x: -x[1])][:K_REC]
    recs_apriori[c] = completar(lista, relleno_val)
print(f"\nCompradores con al menos una regla activada: {con_reglas:,} "
      f"({100 * con_reglas / len(clientes_eval):.1f} %)")
print(registrar(recs_apriori, "Apriori"))

# ------------------------------------------------------------
# 8.6 Filtrado colaborativo basado en usuarios
# ------------------------------------------------------------
cand_lista = np.array(sorted(cand_val))
col_cand = pd.Series(np.arange(len(cand_lista)), index=cand_lista)

def matriz_interaccion(hist, n_clientes):
    """Matriz binaria dispersa clientes × candidatos."""
    h = hist[hist["product_code"].isin(col_cand.index)][["cliente", "product_code"]].drop_duplicates()
    return csr_matrix((np.ones(len(h), dtype=np.float32),
                       (h["cliente"].to_numpy(), col_cand[h["product_code"]].to_numpy())),
                      shape=(n_clientes, len(cand_lista)))

R_val = matriz_interaccion(hist_val, len(muestra))
print(f"\nMatriz R: {R_val.shape[0]:,} × {R_val.shape[1]:,} | Entradas no nulas: {R_val.nnz:,} "
      f"| Densidad: {100 * R_val.nnz / np.prod(R_val.shape):.4f} %")
R_norm = normalize(R_val, norm="l2", axis=1)

def top_productos(puntajes, clientes_lote, recs):
    """Agrega a recs los 12 productos de mayor puntaje positivo de cada cliente del lote."""
    orden = np.argsort(-puntajes, axis=1)[:, :K_REC]
    for j, c in enumerate(clientes_lote):
        pos = orden[j][puntajes[j, orden[j]] > 0]
        recs[c] = completar(cand_lista[pos].tolist(), relleno_val)

def recs_cf_usuarios(K, lote=500):
    """Filtrado colaborativo con los K vecinos de mayor similitud de coseno."""
    recs = {}
    for i in range(0, len(clientes_eval), lote):
        cl = clientes_eval[i:i + lote]
        S = (R_norm[cl] @ R_norm.T).toarray()
        S[np.arange(len(cl)), cl] = 0                         # Excluye al propio cliente
        vecinos = np.argpartition(-S, K, axis=1)[:, :K]
        pesos = np.take_along_axis(S, vecinos, axis=1)
        W = csr_matrix((pesos.ravel(), (np.repeat(np.arange(len(cl)), K), vecinos.ravel())),
                       shape=(len(cl), R_val.shape[0]))
        top_productos((W @ R_val).toarray(), cl, recs)
    return recs

for K in [20, 50, 100]:
    t0 = time.time()
    print(registrar(recs_cf_usuarios(K), f"CF usuarios (K={K})"), f"| {time.time() - t0:.1f} s")

# ------------------------------------------------------------
# 8.7 Factorización de matrices (SVD)
# ------------------------------------------------------------
t0 = time.time()
U, s, Vt = svds(R_val, k=128, random_state=SEED)
orden_sv = np.argsort(-s)                                     # svds devuelve en orden ascendente
U, s, Vt = U[:, orden_sv], s[orden_sv], Vt[orden_sv]
print(f"\nSVD (d = 128): {time.time() - t0:.1f} s")

for d in [32, 64, 128]:
    recs = {}
    for i in range(0, len(clientes_eval), 1000):
        cl = clientes_eval[i:i + 1000]
        top_productos((U[cl, :d] * s[:d]) @ Vt[:d], cl, recs)
    print(registrar(recs, f"SVD (d={d})"))

# Valores singulares: proporción de la energía capturada
energia = np.cumsum(s**2) / (R_val.multiply(R_val)).sum()
print(f"Proporción de la norma de Frobenius² capturada: d=32 {energia[31]:.3f} | "
      f"d=64 {energia[63]:.3f} | d=128 {energia[127]:.3f}")

pd.DataFrame(resultados_val).round(5)

**Significancia de las diferencias.** El historial del cliente supera a la popularidad global de forma significativa: la diferencia de MAP@12 es de 0,0257, con un intervalo de confianza del 95 % entre 0,0220 y 0,0293. En cambio, las mejoras de los segmentos de K-medoides (+0,0006; IC: −0,0003 a 0,0015) y de los grupos de edad (+0,0003; IC: −0,0011 a 0,0015) no son significativas, y tampoco lo es la diferencia de K-medias con $k = 7$ (−0,0010; IC: −0,0021 a 0,0001). Solo K-medias con $k = 3$ es significativamente inferior a la popularidad global (−0,0028; IC: −0,0043 a −0,0014). Esto matiza lo observado en la subsección 8.3: no existe evidencia estadística de que la segmentación mejore la popularidad global, aunque sí de que una segmentación basada en la actividad la empeora.

**Reglas de asociación.** De las 70.429 canastas de las últimas 8 semanas, 45.133 contienen dos o más productos. Con un soporte mínimo de 0,0005 (unas 35 canastas), Apriori obtiene 128 reglas en 6,2 segundos. Las reglas de mayor *lift* asocian las dos piezas de un mismo conjunto, como la parte superior e inferior de un traje de baño (Bellini Bandeau y Bellini Cheeky Brief, *lift* de 921) o de un conjunto de ropa interior (Kanye Vest Top y Kanye Vest Mid Brief, *lift* de 735). Las reglas se activan para el 55,9 % de los compradores evaluados, pero el MAP@12 resultante (0,0183) es 7,6 % inferior al de la popularidad global. Una posible explicación es que las reglas capturan complementos que suelen comprarse en la misma visita: cuando el cliente compra una pieza del conjunto, normalmente ya compró también la otra, de modo que la regla recomienda algo que ya tiene. Esta interpretación es una hipótesis.

**Filtrado colaborativo basado en usuarios.** El MAP@12 aumenta con el número de vecinos: 0,0233 con $K = 20$, 0,0243 con $K = 50$ y 0,0254 con $K = 100$, un 28 % por encima de la popularidad global. Sin embargo, su Precisión@12 (0,0114) y su Recall@12 (0,0587) son ligeramente inferiores a los de la popularidad. El filtrado colaborativo no acierta más productos, sino que ubica los aciertos en posiciones más altas de la lista, lo que el MAP premia. Es también el método más diverso después del historial (cobertura entre 26 % y 31 %). Como el mejor valor se obtiene en el extremo de la grilla, conviene explorar valores mayores de $K$.

**Factorización de matrices.** El SVD obtiene un MAP@12 entre 0,0209 y 0,0214, levemente superior a la popularidad, pero con menor precisión y recall, que además disminuyen al aumentar la dimensión latente. Los 128 primeros valores singulares capturan solo el 18,1 % de la norma de Frobenius al cuadrado de la matriz (8,6 % con $d = 32$): la matriz es tan dispersa (0,07 % de entradas no nulas) que una aproximación de bajo rango retiene poca de su estructura. La baja cobertura (entre 1,3 % y 3,6 %) sugiere que las primeras dimensiones latentes reproducen sobre todo la popularidad de los productos, lo que es esperable en una descomposición sin centrar.

**Interpretación.** El filtrado colaborativo y el SVD se entrenan con los dos años de historia y dan el mismo peso a una compra de 2018 que a una de la semana anterior, mientras que la popularidad funciona mejor cuanto más corta es su ventana (subsección 8.2). Esta tensión sugiere que restringir la historia a las semanas recientes podría mejorar los métodos colaborativos, lo que se evalúa a continuación.

#### **8.8 Recomendación basada en contenido**

Siguiendo los recomendadores clásicos por trama y por metadatos, cada producto candidato se representa mediante dos bloques de texto. El primero es la descripción del producto (`detail_desc`, en inglés), limpiada con expresiones regulares (minúsculas, sin números ni signos de puntuación) y vectorizada con TF-IDF, eliminando las palabras vacías del inglés y los términos que aparecen en un solo producto. El segundo son los metadatos de clasificación (tipo y grupo de producto, grupo de índice, índice, sección y grupo de prenda), vectorizados con `CountVectorizer` como indicadores binarios. Ambos bloques se normalizan, se concatenan y se vuelven a normalizar, de modo que el producto interno entre dos productos equivale a su similitud de coseno. El perfil de cada cliente es el promedio de los vectores de los 20 productos que compró más recientemente, y se le recomiendan los candidatos de mayor similitud con ese perfil.

#### **8.9 Máquinas de Boltzmann restringidas**

Se entrena una máquina de Boltzmann restringida de Bernoulli (`BernoulliRBM`) sobre la matriz binaria de clientes × productos, con una unidad visible por producto. Para mantener un tiempo de entrenamiento razonable, las unidades visibles se restringen a los 3.000 productos candidatos más vendidos en las últimas cuatro semanas de entrenamiento, y se usan lotes de 256 clientes. Los puntajes de recomendación son las probabilidades de activación de las unidades visibles reconstruidas a partir de las unidades ocultas. Se exploran 64 y 128 unidades ocultas y tasas de aprendizaje de 0,05 y 0,10, con 20 iteraciones. A diferencia de la implementación habitual, la configuración se elige por el MAP@12 en validación y no por el error de reconstrucción, que se reporta como referencia: el objetivo del proyecto es ordenar productos para cada cliente, no reconstruir la matriz de compras.

#### **8.10 Sensibilidad temporal del filtrado colaborativo**

Para evaluar si la antigüedad de las compras perjudica a los métodos colaborativos, el filtrado colaborativo basado en usuarios se reentrena con $K = 100$ usando solo las compras de las últimas 12 y 26 semanas. Se evalúa además $K = 200$ con toda la historia, pues el mejor valor de la grilla anterior se obtuvo en su extremo.

#### **8.11 Síntesis de los recomendadores individuales**

Se reúnen los resultados de todos los recomendadores y se compara la mejor configuración de cada familia contra la popularidad global mediante intervalos *bootstrap* pareados.

In [ ]:
# ============================================================
# 8.8 – 8.11 CONTENIDO, RBM, SENSIBILIDAD TEMPORAL Y SÍNTESIS
# ============================================================

import re
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.neural_network import BernoulliRBM
from sklearn.metrics import mean_squared_error

def top_productos(puntajes, clientes_lote, recs, items=None):
    """Agrega a recs los 12 productos de mayor puntaje positivo de cada cliente del lote."""
    items = cand_lista if items is None else items
    orden = np.argsort(-puntajes, axis=1)[:, :K_REC]
    for j, c in enumerate(clientes_lote):
        pos = orden[j][puntajes[j, orden[j]] > 0]
        recs[c] = completar(items[pos].tolist(), relleno_val)

# ------------------------------------------------------------
# 8.8 Recomendación basada en contenido
# ------------------------------------------------------------
def limpiar(txt):
    """Minúsculas, sin enlaces, números ni puntuación."""
    if pd.isna(txt):
        return ""
    txt = re.sub(r"http\S+", " ", str(txt).lower())
    txt = re.sub(r"[^a-z\s]", " ", txt)
    return re.sub(r"\s+", " ", txt).strip()

prod_cand = prod.loc[cand_lista]
desc = prod_cand["detail_desc"].map(limpiar)
cols_meta = ["product_type_name", "product_group_name", "index_group_name",
             "index_name", "section_name", "garment_group_name"]
meta = prod_cand[cols_meta].fillna("").apply(
    lambda r: " ".join(re.sub(r"[^a-z0-9]", "", str(v).lower()) for v in r), axis=1)

tfidf = TfidfVectorizer(stop_words="english", min_df=2)
X_desc = tfidf.fit_transform(desc)
X_meta = CountVectorizer(binary=True).fit_transform(meta)
X_item = normalize(hstack([normalize(X_desc), normalize(X_meta)]).tocsr())
print(f"Vocabulario de descripciones: {X_desc.shape[1]:,} términos | "
      f"Metadatos: {X_meta.shape[1]:,} categorías | Matriz de productos: {X_item.shape}")

# Perfil de cada comprador evaluado: promedio de sus 20 productos más recientes
filas, cols = [], []
for i, c in enumerate(clientes_eval):
    for p in recientes.get(c, []):
        filas.append(i)
        cols.append(col_cand[p])
P0 = csr_matrix((np.ones(len(filas), dtype=np.float32), (filas, cols)),
                shape=(len(clientes_eval), len(cand_lista)))
perfil_cont = normalize(P0 @ X_item)

recs_cont = {}
for i in range(0, len(clientes_eval), 1000):
    top_productos((perfil_cont[i:i + 1000] @ X_item.T).toarray(), clientes_eval[i:i + 1000], recs_cont)
print(registrar(recs_cont, "Contenido (TF-IDF + metadatos)"))

# Ejemplo ilustrativo para un cliente con historial
ej = next(c for c in clientes_eval if len(recientes.get(c, [])) >= 5)
print(f"\nCliente {ej}: últimos productos comprados")
print(prod.loc[recientes[ej][:5], ["prod_name", "product_type_name"]].to_string())
print("Recomendaciones por contenido:")
print(prod.loc[recs_cont[ej][:5], ["prod_name", "product_type_name"]].to_string())

# ------------------------------------------------------------
# 8.9 Máquinas de Boltzmann restringidas
# ------------------------------------------------------------
N_RBM = 3000
ventas4 = hist_val[(hist_val["fecha"] > corte_val - pd.Timedelta(weeks=4))
                   & hist_val["product_code"].isin(cand_val)]["product_code"].value_counts()
items_rbm = np.sort(ventas4.index[:N_RBM].to_numpy())
X_rbm = R_val[:, col_cand[items_rbm].to_numpy()]
X_train_rbm = X_rbm[np.where(X_rbm.getnnz(axis=1) > 0)[0]]

pares_val = obj_val[["cliente", "product_code"]].drop_duplicates()
print(f"\nRBM: {X_train_rbm.shape[0]:,} clientes × {N_RBM:,} productos | "
      f"Techo de recall con estos productos: {100 * pares_val['product_code'].isin(items_rbm).mean():.2f} %")

def reconstruir(rbm, X):
    """Probabilidades de las unidades visibles reconstruidas."""
    v = rbm.transform(X) @ rbm.components_ + rbm.intercept_visible_
    return 1 / (1 + np.exp(-v))

idx_err = np.random.default_rng(SEED).choice(X_train_rbm.shape[0], 5000, replace=False)
X_err = X_train_rbm[idx_err]
resultados_rbm = []
for nc in [64, 128]:
    for lr in [0.05, 0.10]:
        t0 = time.time()
        rbm = BernoulliRBM(n_components=nc, learning_rate=lr, n_iter=20, batch_size=256,
                           random_state=SEED, verbose=0).fit(X_train_rbm)
        error = mean_squared_error(X_err.toarray(), reconstruir(rbm, X_err))
        recs = {}
        for i in range(0, len(clientes_eval), 1000):
            cl = clientes_eval[i:i + 1000]
            top_productos(reconstruir(rbm, X_rbm[cl]), cl, recs, items=items_rbm)
        fila = registrar(recs, f"RBM (ocultas={nc}, lr={lr})")
        resultados_rbm.append({**fila, "error_reconstrucción": error, "segundos": time.time() - t0})
        print(fila["modelo"], f"| MAP@12 = {fila['MAP@12']:.5f} | error = {error:.5f} | {time.time() - t0:.1f} s")

# ------------------------------------------------------------
# 8.10 Sensibilidad temporal del filtrado colaborativo
# ------------------------------------------------------------
def cf_usuarios(K, R, lote=500):
    """Filtrado colaborativo basado en usuarios sobre la matriz R."""
    Rn = normalize(R, norm="l2", axis=1)
    recs = {}
    for i in range(0, len(clientes_eval), lote):
        cl = clientes_eval[i:i + lote]
        S = (Rn[cl] @ Rn.T).toarray()
        S[np.arange(len(cl)), cl] = 0
        vecinos = np.argpartition(-S, K, axis=1)[:, :K]
        pesos = np.take_along_axis(S, vecinos, axis=1)
        W = csr_matrix((pesos.ravel(), (np.repeat(np.arange(len(cl)), K), vecinos.ravel())),
                       shape=(len(cl), R.shape[0]))
        top_productos((W @ R).toarray(), cl, recs)
    return recs

print()
print(registrar(cf_usuarios(200, R_val), "CF usuarios (K=200)"))
for sem in [26, 12]:
    R_sem = matriz_interaccion(hist_val[hist_val["fecha"] > corte_val - pd.Timedelta(weeks=sem)], len(muestra))
    print(registrar(cf_usuarios(100, R_sem), f"CF usuarios (K=100, {sem} sem.)"))

# ------------------------------------------------------------
# 8.11 Síntesis de los recomendadores individuales
# ------------------------------------------------------------
tabla_val = pd.DataFrame(resultados_val).sort_values("MAP@12", ascending=False)
print("\nResultados en validación (ordenados por MAP@12):")
print(tabla_val.round(5).to_string(index=False))

def mejor_de(prefijo):
    return tabla_val[tabla_val["modelo"].str.startswith(prefijo)].iloc[0]["modelo"]

familias = ["Historial del cliente", mejor_de("CF usuarios"), mejor_de("SVD"), mejor_de("RBM"),
            "Contenido (TF-IDF + metadatos)", "Segmento: K-medoides (k=3)", "Apriori"]
print("\nMejor configuración de cada familia frente a la popularidad global:")
print(pd.DataFrame([comparar_pareado(m, REF) for m in familias]).round(5).to_string(index=False))

**Recomendación basada en contenido.** Los productos candidatos quedan representados por 1.834 dimensiones: 1.616 términos de las descripciones y 218 categorías de metadatos. El recomendador obtiene un MAP@12 de 0,0119, un 40 % inferior al de la popularidad global, con una diferencia significativa (−0,0079; IC: −0,0103 a −0,0055). Su cobertura es alta (45,8 %), pero la precisión es la más baja de los métodos evaluados (0,0052). El ejemplo ilustra la limitación: a un cliente cuyas compras recientes incluyen zapatillas, bailarinas, una gabardina y un vestido gabardina, el modelo le recomienda camisas y vestidos, productos textualmente similares a parte de su historial pero sin relación con su interés por el calzado. El filtrado por contenido no considera lo que compran otros clientes ni la popularidad de los productos: recomienda artículos parecidos a los ya comprados, aunque nadie los esté adquiriendo, lo que resulta desfavorable en un mercado dominado por tendencias de corta duración.

**Máquinas de Boltzmann restringidas.** Con los 3.000 productos más vendidos del último mes, la RBM se entrena sobre 107.700 clientes y su techo de recall es del 81,4 %. Todas las configuraciones son significativamente inferiores a la popularidad global: la mejor (64 unidades ocultas y tasa de 0,05) obtiene un MAP@12 de 0,0114 (−0,0084; IC: −0,0105 a −0,0065), y el desempeño empeora con más unidades ocultas y con una tasa de aprendizaje mayor. Además, el error de reconstrucción es idéntico en las cuatro configuraciones (0,00259): en una matriz con 99,9 % de ceros, el error está dominado por la predicción de las entradas vacías y no discrimina entre modelos, lo que justifica haber seleccionado la configuración por el MAP y no por el error de reconstrucción. Su cobertura es mínima (cerca de 0,1 %), pues concentra las recomendaciones en pocos productos populares, y cada configuración tardó entre 7 y 8 minutos. Como el filtrado colaborativo inicial, la RBM se entrenó con toda la historia, lo que podría explicar parte de su bajo desempeño; esta posibilidad no se evaluó por su costo computacional.

**Sensibilidad temporal del filtrado colaborativo.** Ampliar el vecindario a $K = 200$ mejora poco el MAP@12 (de 0,0254 a 0,0260). Restringir la historia a las compras recientes tiene, en cambio, un efecto marcado: con las últimas 26 semanas el MAP@12 sube a 0,0315 (+24 % frente a la historia completa con el mismo $K$) y con las últimas 12 semanas a 0,0358 (+41 %). Con 12 semanas, la Precisión@12 (0,0144) y el Recall@12 (0,0779) superan por primera vez a los de la popularidad global. Este resultado confirma que la vigencia de la información es determinante en este dominio: los vecinos más útiles son los clientes que compraron productos similares recientemente, no los que coincidieron con el cliente hace uno o dos años.

**Síntesis.** Solo dos recomendadores superan a la popularidad global con significancia estadística: el **historial del cliente** (+0,0257; IC: 0,0220 a 0,0293) y el **filtrado colaborativo con historia de 12 semanas** (+0,0160; IC: 0,0131 a 0,0190). Las diferencias del SVD (+0,0016), los segmentos de K-medoides (+0,0006) y las reglas de asociación (−0,0015) no son significativas, mientras que el contenido y la RBM son significativamente inferiores. Las dos señales que funcionan son complementarias: el historial captura lo que el cliente vuelve a comprar, y el filtrado colaborativo reciente anticipa productos nuevos para él a partir de clientes con compras similares. Ambas, junto con la popularidad de la última semana como respaldo, constituyen la base del modelo híbrido de la sección 9.

### **9. Sistema de recomendación híbrido**

#### **9.1 Estrategia de combinación**

El modelo híbrido combina los recomendadores individuales mediante una ponderación, una de las estrategias de hibridación clasificadas por Burke (2002). Como los puntajes de cada método se expresan en escalas no comparables (similitudes, conteos o probabilidades), la combinación no se hace sobre los puntajes sino sobre las posiciones, mediante la **fusión por rango recíproco ponderada** (Cormack, Clarke y Büttcher, 2009). Cada método $m$ genera para cada cliente una lista de hasta 50 productos, y el puntaje híbrido de un producto $p$ es

$$s(p) = \sum_m \frac{w_m}{60 + r_m(p)},$$

donde $r_m(p)$ es la posición del producto en la lista del método $m$ (el término no se suma si el producto no aparece en ella) y $w_m$ es el peso del método. La constante 60 es la propuesta por los autores y atenúa la diferencia entre las primeras posiciones. Se recomiendan los 12 productos de mayor puntaje y, si son menos de 12, la lista se completa con los más populares.

Los componentes son las tres señales con desempeño significativo o equivalente al de la referencia (historial del cliente, filtrado colaborativo con historia de 12 semanas y popularidad de la última semana), más dos señales que, aunque no superan a la popularidad por sí solas, podrían aportar información complementaria: los segmentos de K-medoides y la similitud de contenido.

#### **9.2 Calibración de los pesos**

Con el peso de la popularidad fijo en uno, los pesos de los demás componentes se eligen por búsqueda en grilla sobre la semana de validación: historial y filtrado colaborativo en $\{0, 1, 2, 4\}$, segmento y contenido en $\{0;\ 0{,}5;\ 1\}$, para un total de 144 combinaciones. Como los pesos se ajustan sobre la misma semana en que se miden, el desempeño en validación es optimista; la estimación sin sesgo se obtiene en la semana de prueba (sección 10).

#### **9.3 Análisis de ablación**

Para cuantificar el aporte de cada componente, a partir de la mejor combinación se elimina un componente a la vez (peso cero) y se mide la caída del MAP@12, con su intervalo de confianza *bootstrap* pareado.

In [ ]:
# ============================================================
# 9. SISTEMA DE RECOMENDACIÓN HÍBRIDO
# ============================================================

import itertools

N_FUS, K_RRF = 50, 60
col_dict = dict(zip(cand_lista, range(len(cand_lista))))
pos_eval = {c: i for i, c in enumerate(clientes_eval)}

# ------------------------------------------------------------
# 9.1 Listas puras (sin relleno) de cada componente
# ------------------------------------------------------------
def top_n_listas(puntajes, clientes_lote, destino, items=None, n=N_FUS):
    """Top-n productos con puntaje positivo, sin completar con populares."""
    items = cand_lista if items is None else items
    orden = np.argsort(-puntajes, axis=1)[:, :n]
    for j, c in enumerate(clientes_lote):
        pos = orden[j][puntajes[j, orden[j]] > 0]
        destino[c] = items[pos].tolist()

# Historial del cliente
L_hist = hist_cli.groupby("cliente")["product_code"].apply(lambda s: s.head(N_FUS).tolist()).to_dict()

# Popularidad de la última semana
top_pop = populares(hist_val, corte_val, W_POP, cand_val, n=N_FUS)
L_pop = {c: top_pop for c in clientes_eval}

# Segmentos de K-medoides (respaldo de edad para clientes sin perfil)
def listas_por_grupo(grupo_cliente, n=N_FUS):
    h = hist_val[(hist_val["fecha"] > corte_val - pd.Timedelta(weeks=W_POP))
                 & hist_val["product_code"].isin(cand_val)]
    h = h.assign(grupo=h["cliente"].map(grupo_cliente))
    top = h.groupby("grupo")["product_code"].value_counts().groupby(level=0).head(n)
    listas = top.reset_index().groupby("grupo")["product_code"].apply(list).to_dict()
    return {c: listas.get(grupo_cliente.get(c), []) for c in clientes_eval}
L_seg = listas_por_grupo(grupos_con_respaldo(particiones_seg["K-medoides (k=3)"]))

# Filtrado colaborativo con historia de 12 semanas (K = 100)
R_12 = matriz_interaccion(hist_val[hist_val["fecha"] > corte_val - pd.Timedelta(weeks=12)], len(muestra))
R12n = normalize(R_12, norm="l2", axis=1)
L_cf = {}
for i in range(0, len(clientes_eval), 500):
    cl = clientes_eval[i:i + 500]
    S = (R12n[cl] @ R12n.T).toarray()
    S[np.arange(len(cl)), cl] = 0
    vecinos = np.argpartition(-S, 100, axis=1)[:, :100]
    pesos = np.take_along_axis(S, vecinos, axis=1)
    W = csr_matrix((pesos.ravel(), (np.repeat(np.arange(len(cl)), 100), vecinos.ravel())),
                   shape=(len(cl), R_12.shape[0]))
    top_n_listas((W @ R_12).toarray(), cl, L_cf)

# Contenido
L_cont = {}
for i in range(0, len(clientes_eval), 1000):
    top_n_listas((perfil_cont[i:i + 1000] @ X_item.T).toarray(), clientes_eval[i:i + 1000], L_cont)

def matriz_rrf(listas):
    """Matriz dispersa clientes evaluados × candidatos con 1 / (60 + posición)."""
    filas, cols, vals = [], [], []
    for c, lista in listas.items():
        i = pos_eval[c]
        for r, p in enumerate(lista[:N_FUS]):
            j = col_dict.get(p)
            if j is not None:
                filas.append(i); cols.append(j); vals.append(1 / (K_RRF + r + 1))
    return csr_matrix((np.array(vals, dtype=np.float32), (filas, cols)),
                      shape=(len(clientes_eval), len(cand_lista)))

M = {"historial": matriz_rrf(L_hist), "cf": matriz_rrf(L_cf), "popularidad": matriz_rrf(L_pop),
     "segmento": matriz_rrf(L_seg), "contenido": matriz_rrf(L_cont)}
for nombre, m in M.items():
    print(f"{nombre:12s}: clientes con lista = {(m.getnnz(axis=1) > 0).sum():,} | "
          f"productos por cliente (media) = {m.getnnz(axis=1).mean():.1f}")

def recs_desde_matriz(H):
    """Top-12 de la matriz híbrida, completando con populares."""
    H = H.tocsr()
    recs = {}
    for i, c in enumerate(clientes_eval):
        a, b = H.indptr[i], H.indptr[i + 1]
        lista = cand_lista[H.indices[a:b][np.argsort(-H.data[a:b])[:K_REC]]].tolist() if b > a else []
        recs[c] = completar(lista, relleno_val)
    return recs

def hibrido(pesos):
    H = sum(w * M[m] for m, w in pesos.items() if w > 0)
    return recs_desde_matriz(H)

# ------------------------------------------------------------
# 9.2 Calibración de los pesos (búsqueda en grilla)
# ------------------------------------------------------------
t0 = time.time()
grilla = []
for wh, wc, ws, wt in itertools.product([0, 1, 2, 4], [0, 1, 2, 4], [0, 0.5, 1], [0, 0.5, 1]):
    pesos = {"historial": wh, "cf": wc, "popularidad": 1, "segmento": ws, "contenido": wt}
    fila, _ = evaluar(hibrido(pesos), verdad_val, cand_val, "Híbrido")
    grilla.append({**{f"w_{k}": v for k, v in pesos.items()}, **{k: fila[k] for k in
                   ["MAP@12", "Precisión@12", "Recall@12", "cobertura_%"]}})
grilla = pd.DataFrame(grilla).sort_values("MAP@12", ascending=False)
print(f"\nBúsqueda en grilla: {len(grilla)} combinaciones en {time.time() - t0:.1f} s")
print("Diez mejores combinaciones:")
print(grilla.head(10).round(5).to_string(index=False))

mejor = grilla.iloc[0]
pesos_opt = {k.replace("w_", ""): float(mejor[k]) for k in grilla.columns if k.startswith("w_")}
print(f"\nPesos elegidos: {pesos_opt}")
print(registrar(hibrido(pesos_opt), "Híbrido ponderado"))

# Comparación del híbrido con sus mejores componentes
print("\nHíbrido frente a los componentes individuales:")
print(pd.DataFrame([comparar_pareado("Híbrido ponderado", m) for m in
                    ["Historial del cliente", "CF usuarios (K=100, 12 sem.)", REF]])
      .round(5).to_string(index=False))

# ------------------------------------------------------------
# 9.3 Análisis de ablación
# ------------------------------------------------------------
ablacion = []
for comp in [k for k, w in pesos_opt.items() if w > 0]:
    p = {**pesos_opt, comp: 0}
    if sum(p.values()) == 0:
        continue
    nombre = f"Híbrido sin {comp}"
    registrar(hibrido(p), nombre)
    ablacion.append(comparar_pareado("Híbrido ponderado", nombre))
print("\nAblación (caída del MAP@12 al eliminar cada componente):")
print(pd.DataFrame(ablacion).round(5).to_string(index=False))

**Disponibilidad de las señales.** No todos los componentes generan lista para todos los compradores. El historial y el contenido la generan para 6.651 clientes (92,3 %), con una media de 26,7 productos en el historial; el filtrado colaborativo solo para 5.241 (72,7 %), los que compraron en las últimas 12 semanas; y el segmento para 6.670, pues en las listas de fusión los clientes sin historia quedaron sin el respaldo de edad. Solo la popularidad cubre a todos los compradores, lo que la convierte en el respaldo natural del híbrido.

**Calibración de los pesos.** La búsqueda en grilla (144 combinaciones en 45,3 segundos) selecciona un peso de 4 para el historial, de 1 para la popularidad y de cero para el filtrado colaborativo, el segmento y el contenido, con un MAP@12 de 0,0476. Las cinco mejores combinaciones excluyen el filtrado colaborativo, y agregar el segmento con peso 0,5 aumenta levemente la precisión (0,0175) y el recall (0,0947) pero reduce el MAP (0,0474). Las combinaciones que incluyen el filtrado colaborativo alcanzan valores de MAP de a lo sumo 0,0461, aunque algunas logran la mayor precisión y el mayor recall de la grilla (0,0183 y 0,0963 con historial 2, filtrado colaborativo 1 y segmento 1), a costa de una menor cobertura.

**Interpretación de los pesos.** Con un peso de 2 o más para el historial, cualquier producto del historial del cliente (posiciones 1 a 50) obtiene un puntaje mayor que el producto más popular, pues $2/(60+50) > 1/61$. El híbrido conserva, por tanto, el historial en las primeras posiciones, pero **reordenado según la popularidad actual**: los productos que el cliente ya compró y que además están entre los más vendidos de la última semana reciben un puntaje adicional y suben en la lista. Los populares completan la lista solo cuando el historial tiene menos de 12 productos. Por esta razón los pesos 2 y 4 producen resultados similares, y aumentar el peso por encima del límite de la grilla solo reduciría el efecto de la popularidad, acercando el híbrido al historial puro.

**Desempeño frente a los componentes.** El híbrido supera de forma significativa al historial del cliente (+0,0021; IC: 0,0004 a 0,0040; un 4,6 % de mejora), al filtrado colaborativo de 12 semanas (+0,0118; IC: 0,0093 a 0,0144) y a la popularidad global (+0,0278; IC: 0,0244 a 0,0314). El análisis de ablación confirma que el historial es el componente central: sin él, el MAP@12 cae al nivel de la popularidad (−0,0278), mientras que sin la popularidad cae al nivel del historial (−0,0021).

**Por qué el filtrado colaborativo no entra en el híbrido.** Aunque por sí solo es el segundo mejor recomendador, su aporte desaparece al combinarlo con el historial. Una posible explicación es que el filtrado colaborativo no excluye los productos que el cliente ya compró, y parte de su ventaja sobre la popularidad proviene de recomendar justamente esos productos, que en el híbrido ya aporta el historial. Para los productos nuevos para el cliente, sus listas serían menos precisas que los más vendidos de la semana. Esta hipótesis se verifica en la sección 10, midiendo qué proporción de los aciertos de cada modelo corresponde a recompras.

**Conclusión.** En validación, el mejor sistema combina dos señales simples: lo que el cliente ya compró y lo que se está vendiendo ahora. Las señales derivadas de la segmentación, el filtrado colaborativo y el contenido no mejoran el MAP@12 en esta combinación, aunque las configuraciones que incluyen el filtrado colaborativo ofrecen un balance distinto entre precisión, recall y cobertura. Por ello, en la sección 10 se evalúan en la semana de prueba tanto el híbrido seleccionado como la mejor combinación que incluye el filtrado colaborativo.

### **10. Evaluación comparativa**

#### **10.1 Evaluación en la semana de prueba**

Todos los recomendadores se reentrenan con la historia hasta el 15 de septiembre de 2020 y se evalúan con las compras de la semana del 16 al 22 de septiembre, que no intervino en ninguna decisión previa. Se mantienen las configuraciones elegidas en validación: ventana de una semana para la popularidad, $K = 100$ vecinos con historia de 12 semanas para el filtrado colaborativo, $d = 128$ para el SVD, la mejor configuración de la RBM y los pesos del híbrido. Los segmentos de los clientes se recalculan con su historia actualizada, aplicando el escalador, el ACP y los medoides de K-medoides ya ajustados, sin volver a estimarlos. Se evalúan dos versiones del híbrido: la seleccionada en validación (historial y popularidad) y la mejor combinación que incluye el filtrado colaborativo (historial 4, filtrado colaborativo 1, popularidad 1 y segmento 0,5).

#### **10.2 Estabilidad entre validación y prueba**

Se compara el MAP@12 de cada modelo en ambas semanas. Como los pesos del híbrido se calibraron en validación, una caída de su desempeño en prueba mayor que la de los demás modelos indicaría sobreajuste de la calibración.

#### **10.3 Desempeño por tipo de cliente**

El MAP@12 se desagrega según el nivel de actividad de los clientes (estratos de la sección 3) y se separa a los compradores sin historia, para identificar a qué clientes beneficia cada recomendador.

#### **10.4 Origen de los aciertos**

Para verificar la hipótesis planteada en la sección 9, se calcula qué proporción de los aciertos de cada modelo corresponde a **recompras** (productos que el cliente ya había comprado) y qué proporción a productos nuevos para él.


In [ ]:
# ============================================================
# 10. EVALUACIÓN COMPARATIVA
# ============================================================

EVALUAR_RBM = True
pesos_cf = {"historial": 4, "cf": 1, "popularidad": 1, "segmento": 0.5, "contenido": 0}
registrar(hibrido(pesos_cf), "Híbrido con CF")           # Referencia en validación

def comparar_pareado(a, b, aps=None, B=2000, seed=SEED):
    """IC del 95 % bootstrap para la diferencia media de AP@12 entre dos modelos."""
    aps = ap_val if aps is None else aps
    d = aps[a] - aps[b]
    rng_b = np.random.default_rng(seed)
    boot = np.array([d[rng_b.integers(0, len(d), len(d))].mean() for _ in range(B)])
    li, ls = np.percentile(boot, [2.5, 97.5])
    return {"comparación": f"{a} − {b}", "diferencia_MAP": d.mean(),
            "IC95_inf": li, "IC95_sup": ls, "significativa": not (li <= 0 <= ls)}

# ------------------------------------------------------------
# 10.1 Contexto del escenario de prueba
# ------------------------------------------------------------
hist_t = tx_m[tx_m["fecha"] <= corte_test]
obj_t  = tx_m[tx_m["fecha"] > corte_test]
cand_t = escenarios["Prueba"]["candidatos"]
cand_lista_t = np.array(sorted(cand_t))
col_t = dict(zip(cand_lista_t, range(len(cand_lista_t))))
verdad_t = obj_t.groupby("cliente")["product_code"].apply(set).to_dict()
clientes_t = np.array(sorted(verdad_t))
pos_t = {c: i for i, c in enumerate(clientes_t)}
relleno_t = populares(hist_t, corte_test, W_POP, cand_t, n=200)
print(f"Compradores de prueba: {len(clientes_t):,} | Candidatos: {len(cand_lista_t):,}")

resultados_test, ap_test, recs_test = [], {}, {}
def registrar_t(recs, nombre):
    fila, ap = evaluar(recs, verdad_t, cand_t, nombre)
    resultados_test.append(fila); ap_test[nombre] = ap; recs_test[nombre] = recs
    print(f"{nombre:35s} MAP@12 = {fila['MAP@12']:.5f}")

def a_recs(listas):
    """Listas puras → 12 recomendaciones completadas con populares."""
    return {c: completar(listas.get(c, [])[:K_REC], relleno_t) for c in clientes_t}

def top_lista(puntajes, cl, items, n=N_FUS):
    orden = np.argsort(-puntajes, axis=1)[:, :n]
    return {c: items[orden[j][puntajes[j, orden[j]] > 0]].tolist() for j, c in enumerate(cl)}

def matriz_t(hist):
    h = hist[hist["product_code"].isin(cand_t)][["cliente", "product_code"]].drop_duplicates()
    return csr_matrix((np.ones(len(h), dtype=np.float32),
                       (h["cliente"].to_numpy(), h["product_code"].map(col_t).to_numpy())),
                      shape=(len(muestra), len(cand_lista_t)))

# Popularidad
L_pop_t = {c: relleno_t[:N_FUS] for c in clientes_t}
registrar_t(a_recs(L_pop_t), REF)

# Historial del cliente
h = hist_t[hist_t["cliente"].isin(clientes_t) & hist_t["product_code"].isin(cand_t)]
hist_cli_t = (h.groupby(["cliente", "product_code"]).agg(ultima=("fecha", "max"), veces=("fecha", "size"))
               .reset_index().sort_values(["cliente", "ultima", "veces"], ascending=[True, False, False]))
L_hist_t = hist_cli_t.groupby("cliente")["product_code"].apply(lambda s: s.head(N_FUS).tolist()).to_dict()
registrar_t(a_recs(L_hist_t), "Historial del cliente")

# Segmentos de K-medoides con perfiles actualizados (sin reajustar los modelos)
perf_t = perfiles_clientes(corte_test)
perf_t["age"] = perf_t["age"].fillna(edad_mediana)
F_t = pca.transform(escalador.transform(transformar(perf_t)))
seg_t = pd.Series(pairwise_distances(F_t, clara_final["medoides"]).argmin(axis=1), index=perf_t.index)
grupo_t = grupos_con_respaldo(seg_t)
hw = hist_t[(hist_t["fecha"] > corte_test - pd.Timedelta(weeks=W_POP)) & hist_t["product_code"].isin(cand_t)]
def top_por(columna_grupo):
    g = hw.assign(grupo=columna_grupo)
    top = g.groupby("grupo")["product_code"].value_counts().groupby(level=0).head(N_FUS)
    return top.reset_index().groupby("grupo")["product_code"].apply(list).to_dict()
listas_seg = top_por(hw["cliente"].map(grupo_t))
listas_edad = top_por(hw["cliente"].map(edad_cliente))
L_seg_t = {c: listas_seg.get(grupo_t.get(c), listas_edad.get(edad_cliente[c], [])) for c in clientes_t}
registrar_t(a_recs(L_seg_t), "Segmento: K-medoides (k=3)")

# Filtrado colaborativo con historia de 12 semanas
R_t12 = matriz_t(hist_t[hist_t["fecha"] > corte_test - pd.Timedelta(weeks=12)])
Rn_t12 = normalize(R_t12, norm="l2", axis=1)
L_cf_t = {}
for i in range(0, len(clientes_t), 500):
    cl = clientes_t[i:i + 500]
    S = (Rn_t12[cl] @ Rn_t12.T).toarray()
    S[np.arange(len(cl)), cl] = 0
    vec = np.argpartition(-S, 100, axis=1)[:, :100]
    W = csr_matrix((np.take_along_axis(S, vec, axis=1).ravel(),
                    (np.repeat(np.arange(len(cl)), 100), vec.ravel())), shape=(len(cl), R_t12.shape[0]))
    L_cf_t.update(top_lista((W @ R_t12).toarray(), cl, cand_lista_t))
registrar_t(a_recs(L_cf_t), "CF usuarios (K=100, 12 sem.)")

# SVD con toda la historia (d = 128)
R_t = matriz_t(hist_t)
U_t, s_t, Vt_t = svds(R_t, k=128, random_state=SEED)
L_svd_t = {}
for i in range(0, len(clientes_t), 1000):
    cl = clientes_t[i:i + 1000]
    L_svd_t.update(top_lista((U_t[cl] * s_t) @ Vt_t, cl, cand_lista_t))
registrar_t(a_recs(L_svd_t), "SVD (d=128)")

# Apriori (reglas recalculadas con las 8 semanas previas a la prueba)
hc = hist_t[(hist_t["fecha"] > corte_test - pd.Timedelta(weeks=SEM_CANASTA)) & hist_t["product_code"].isin(cand_t)]
canastas_t = hc.groupby(["cliente", "fecha"])["product_code"].apply(lambda s: list(set(s))).tolist()
reglas_t = defaultdict_list = {}
for r in apriori(canastas_t, min_support=MIN_SOPORTE, min_confidence=0.01, min_lift=1.0, max_length=2):
    for o in r.ordered_statistics:
        if len(o.items_base) == 1 and len(o.items_add) == 1:
            reglas_t.setdefault(next(iter(o.items_base)), []).append((next(iter(o.items_add)), o.confidence))
L_apr_t = {}
for c in clientes_t:
    puntaje = {}
    for a in L_hist_t.get(c, [])[:20]:
        for cons, conf in reglas_t.get(a, []):
            puntaje[cons] = puntaje.get(cons, 0) + conf
    L_apr_t[c] = [p for p, _ in sorted(puntaje.items(), key=lambda x: -x[1])]
registrar_t(a_recs(L_apr_t), "Apriori")

# Contenido (representación reconstruida sobre los candidatos de prueba)
prod_t = prod.loc[cand_lista_t]
meta_t = prod_t[cols_meta].fillna("").apply(
    lambda r: " ".join(re.sub(r"[^a-z0-9]", "", str(v).lower()) for v in r), axis=1)
X_item_t = normalize(hstack([
    normalize(TfidfVectorizer(stop_words="english", min_df=2).fit_transform(prod_t["detail_desc"].map(limpiar))),
    normalize(CountVectorizer(binary=True).fit_transform(meta_t))]).tocsr())
filas, cols = [], []
for i, c in enumerate(clientes_t):
    for p in L_hist_t.get(c, [])[:20]:
        filas.append(i); cols.append(col_t[p])
perfil_t = normalize(csr_matrix((np.ones(len(filas), dtype=np.float32), (filas, cols)),
                                shape=(len(clientes_t), len(cand_lista_t))) @ X_item_t)
L_cont_t = {}
for i in range(0, len(clientes_t), 1000):
    L_cont_t.update(top_lista((perfil_t[i:i + 1000] @ X_item_t.T).toarray(), clientes_t[i:i + 1000], cand_lista_t))
registrar_t(a_recs(L_cont_t), "Contenido (TF-IDF + metadatos)")

# RBM con la mejor configuración de validación
nombre_rbm = mejor_de("RBM")
if EVALUAR_RBM:
    nc, lr = re.findall(r"ocultas=(\d+), lr=([\d.]+)", nombre_rbm)[0]
    v4 = hist_t[(hist_t["fecha"] > corte_test - pd.Timedelta(weeks=4))
                & hist_t["product_code"].isin(cand_t)]["product_code"].value_counts()
    items_t = np.sort(v4.index[:N_RBM].to_numpy())
    X_rbm_t = R_t[:, [col_t[p] for p in items_t]]
    t0 = time.time()
    rbm_t = BernoulliRBM(n_components=int(nc), learning_rate=float(lr), n_iter=20, batch_size=256,
                         random_state=SEED).fit(X_rbm_t[np.where(X_rbm_t.getnnz(axis=1) > 0)[0]])
    L_rbm_t = {}
    for i in range(0, len(clientes_t), 1000):
        cl = clientes_t[i:i + 1000]
        L_rbm_t.update(top_lista(reconstruir(rbm_t, X_rbm_t[cl]), cl, items_t))
    registrar_t(a_recs(L_rbm_t), nombre_rbm)
    print(f"   (RBM: {time.time() - t0:.1f} s)")

# Híbridos
def matriz_rrf_t(listas):
    filas, cols, vals = [], [], []
    for c, lista in listas.items():
        for r, p in enumerate(lista[:N_FUS]):
            if p in col_t:
                filas.append(pos_t[c]); cols.append(col_t[p]); vals.append(1 / (K_RRF + r + 1))
    return csr_matrix((np.array(vals, dtype=np.float32), (filas, cols)),
                      shape=(len(clientes_t), len(cand_lista_t)))

M_t = {"historial": matriz_rrf_t(L_hist_t), "cf": matriz_rrf_t(L_cf_t), "popularidad": matriz_rrf_t(L_pop_t),
       "segmento": matriz_rrf_t(L_seg_t), "contenido": matriz_rrf_t(L_cont_t)}
def hibrido_t(pesos):
    H = sum(w * M_t[m] for m, w in pesos.items() if w > 0).tocsr()
    recs = {}
    for i, c in enumerate(clientes_t):
        a, b = H.indptr[i], H.indptr[i + 1]
        lista = cand_lista_t[H.indices[a:b][np.argsort(-H.data[a:b])[:K_REC]]].tolist() if b > a else []
        recs[c] = completar(lista, relleno_t)
    return recs
registrar_t(hibrido_t(pesos_opt), "Híbrido ponderado")
registrar_t(hibrido_t(pesos_cf), "Híbrido con CF")

tabla_test = pd.DataFrame(resultados_test).sort_values("MAP@12", ascending=False)
print("\nResultados en la semana de prueba:")
print(tabla_test.round(5).to_string(index=False))

print("\nComparación con la popularidad global (prueba):")
print(pd.DataFrame([comparar_pareado(m, REF, aps=ap_test) for m in tabla_test["modelo"] if m != REF])
      .round(5).to_string(index=False))
print("\nHíbrido frente a sus alternativas (prueba):")
print(pd.DataFrame([comparar_pareado("Híbrido ponderado", m, aps=ap_test)
                    for m in ["Historial del cliente", "Híbrido con CF", "CF usuarios (K=100, 12 sem.)"]])
      .round(5).to_string(index=False))

# Gráfico: MAP@12 en prueba con intervalo del 95 %
orden_m = tabla_test["modelo"].tolist()[::-1]
medias = [ap_test[m].mean() for m in orden_m]
errores = [1.96 * ap_test[m].std() / np.sqrt(len(ap_test[m])) for m in orden_m]
plt.figure(figsize=(10, 6))
plt.barh(orden_m, medias, xerr=errores, color="steelblue", capsize=3)
plt.axvline(ap_test[REF].mean(), color="gray", linestyle="--", label="Popularidad global")
plt.title("MAP@12 en la semana de prueba (intervalo del 95 %)")
plt.xlabel("MAP@12")
plt.legend()
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 10.2 Estabilidad entre validación y prueba
# ------------------------------------------------------------
val_map = pd.DataFrame(resultados_val).drop_duplicates("modelo", keep="last").set_index("modelo")["MAP@12"]
estab = tabla_test.set_index("modelo")[["MAP@12"]].rename(columns={"MAP@12": "MAP_prueba"})
estab.insert(0, "MAP_validación", val_map.reindex(estab.index))
estab["cambio_%"] = 100 * (estab["MAP_prueba"] / estab["MAP_validación"] - 1)
print("\nMAP@12 en validación y prueba:")
print(estab.round(5))

# ------------------------------------------------------------
# 10.3 Desempeño por tipo de cliente
# ------------------------------------------------------------
con_hist = set(hist_t["cliente"])
act_cliente = muestra.reset_index(drop=True)["estrato_act"].astype(str)
tipo = pd.Series(["Sin historia" if c not in con_hist else act_cliente[c] for c in clientes_t])
modelos_clave = [REF, "Historial del cliente", "CF usuarios (K=100, 12 sem.)", "Híbrido ponderado", "Híbrido con CF"]
por_tipo = pd.DataFrame({m: pd.Series(ap_test[m]).groupby(tipo).mean() for m in modelos_clave})
por_tipo.insert(0, "compradores", tipo.value_counts())
por_tipo = por_tipo.reindex(["Sin historia", "0–2", "3–9", "10–29", "30–99", "100+"])
print("\nMAP@12 por tipo de cliente (prueba):")
print(por_tipo.round(5))

# ------------------------------------------------------------
# 10.4 Origen de los aciertos
# ------------------------------------------------------------
pares_hist_t = set(zip(hist_t["cliente"], hist_t["product_code"]))
origen = []
for m in ["Historial del cliente", "CF usuarios (K=100, 12 sem.)", REF, "Híbrido ponderado", "Híbrido con CF"]:
    recompra = nuevo = 0
    for c, real in verdad_t.items():
        for p in set(recs_test[m][c][:K_REC]) & real:
            if (c, p) in pares_hist_t:
                recompra += 1
            else:
                nuevo += 1
    total = recompra + nuevo
    origen.append({"modelo": m, "aciertos": total, "recompras": recompra, "nuevos": nuevo,
                   "recompras_%": 100 * recompra / total if total else np.nan})
print("\nOrigen de los aciertos (prueba):")
print(pd.DataFrame(origen).round(2).to_string(index=False))

**Desempeño en la semana de prueba.** El historial del cliente obtiene el mayor MAP@12 (0,0461), seguido muy de cerca por el híbrido ponderado (0,0459) y el híbrido con filtrado colaborativo (0,0449); los tres casi triplican a la popularidad global (0,0160). El filtrado colaborativo con historia de 12 semanas ocupa el cuarto lugar (0,0352). Frente a la popularidad, son significativamente superiores los dos híbridos, el historial, el filtrado colaborativo y, a diferencia de lo observado en validación, el SVD (+0,0030; IC: 0,0007 a 0,0053). Los segmentos de K-medoides (+0,0014; IC: −0,0001 a 0,0028) y las reglas de asociación siguen sin diferencias significativas, mientras que el contenido y la RBM son significativamente inferiores.

**El híbrido frente al historial.** En prueba, el híbrido ponderado no se distingue del historial del cliente: la diferencia de MAP@12 es de −0,0001, con un intervalo entre −0,0017 y 0,0016. La ventaja de 0,0021 observada en validación no se reproduce, lo que indica que provenía, al menos en parte, de haber calibrado los pesos sobre la misma semana en que se midieron, el sesgo optimista advertido en la subsección 9.2. El híbrido conserva una precisión (0,0171 frente a 0,0162) y un recall (0,0913 frente a 0,0870) algo mayores, es decir, acierta más productos, pero no los ubica en mejores posiciones. Tampoco hay diferencia significativa entre los dos híbridos (+0,0011; IC: −0,0001 a 0,0023).

**Estabilidad entre validación y prueba.** Los modelos basados en la historia individual son estables: el historial mejora un 1,3 %, el filtrado colaborativo cae un 1,5 % y los híbridos entre 2,6 % y 3,4 %. En cambio, los modelos que dependen de la popularidad agregada pierden entre el 11 % y el 19 % (popularidad −18,9 %, reglas de asociación −16,6 %, segmentos −14,6 % y SVD −11,2 %), y la RBM pierde el 54,2 %. Una posible explicación es que los productos más vendidos de una semana anticipan peor las compras de la siguiente en la semana de prueba que en la de validación, mientras que lo que cada cliente compró antes es una señal más estable. Esta interpretación no se ha verificado con otras semanas.

**Desempeño por tipo de cliente.** Los 584 compradores sin historia (8,5 %) reciben de todos los modelos la misma lista de populares (MAP@12 de 0,0162), excepto en el híbrido con filtrado colaborativo, cuyo componente de segmento les asigna los populares de su grupo de edad y mejora levemente su resultado (0,0179). Entre los clientes con historia, el historial y los híbridos superan ampliamente a la popularidad en todos los niveles de actividad. El híbrido ponderado es el mejor en los niveles 3–9 y 30–99, mientras que el historial puro lo es en los niveles 0–2, 10–29 y 100+, con diferencias pequeñas en todos los casos salvo en 100+ (0,0515 frente a 0,0485). La popularidad pierde eficacia a medida que aumenta la actividad, de 0,0265 en el nivel 0–2 a 0,0129 en el nivel 100+: los clientes más activos compran más productos por semana y sus compras se apartan más de los productos de mayor venta.

**Origen de los aciertos.** El 85,5 % de los aciertos del historial y el 86,3 % de los de ambos híbridos son recompras. El filtrado colaborativo también obtiene la mayoría de sus aciertos por recompra (65,3 %), lo que confirma la hipótesis de la sección 9: su ventaja sobre la popularidad proviene en buena parte de productos que el cliente ya había comprado, información que el historial aporta de forma directa. Sin embargo, el filtrado colaborativo acierta 425 productos nuevos para el cliente, frente a 194 del historial y de los híbridos, y la popularidad acierta 797. Los híbridos no aprovechan esa capacidad: al ubicar siempre el historial en las primeras posiciones, los productos nuevos solo entran en la lista cuando el historial tiene menos de 12 productos.

**Conclusión.** En la semana de prueba, el mejor desempeño se obtiene con la señal más simple, lo que el cliente ya compró, y la combinación con otras señales no la supera de forma significativa. El filtrado colaborativo reciente es el único método no basado en la recompra que mejora de forma sustancial a la popularidad y que aporta aciertos en productos nuevos, aunque el esquema de fusión adoptado no logra integrarlos.

### **11. Extensión: preferencias visuales**

Esta sección evalúa si la apariencia de los productos aporta información sobre las preferencias de los clientes que no está contenida en sus atributos categóricos ni en su descripción textual. Las imágenes se representan mediante *embeddings*, vectores numéricos que resumen el contenido visual de cada fotografía, y se usan para reordenar listas de candidatos que ya son competitivas (los productos más vendidos y las recomendaciones del filtrado colaborativo) según su similitud visual con las compras recientes del cliente. Este diseño responde a lo observado en la sección 8: el filtrado por contenido aplicado de forma aislada ignora la popularidad y resulta inferior a la referencia.

#### **11.1 Selección y descarga de imágenes**

Las imágenes provienen del concurso original de H&M en Kaggle, donde se almacenan por artículo con la ruta `images/<tres primeros dígitos>/<article_id>.jpg`. Para no descargar el catálogo completo, solo se obtienen las imágenes de los productos que intervienen en el reordenamiento: los 100 más vendidos de la última semana, los productos de las listas del filtrado colaborativo y las 20 compras más recientes de cada cliente evaluado, en los escenarios de validación y prueba. Cada `product_code` se representa con la imagen de su variante de color más vendida en la muestra o, si no registra ventas, con la de su primer artículo. Antes de la descarga completa se mide la velocidad con una prueba de 40 imágenes para estimar el tiempo total.

In [ ]:
# Credenciales de Kaggle: guarde su token en los *Secrets* de Colab con el nombre KAGGLE_API_TOKEN
# (nunca lo escriba en el notebook).
from google.colab import userdata
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

# Verificación inmediata
r = subprocess.run(["kaggle", "competitions", "files", "-c", "h-and-m-personalized-fashion-recommendations"],
                   capture_output=True, text=True)
print((r.stdout + r.stderr)[:500])


In [ ]:
# ============================================================
# 11.1 SELECCIÓN Y DESCARGA DE IMÁGENES
# ============================================================

!pip install -q -U kaggle

import os, shutil, zipfile
import concurrent.futures as cfut
from PIL import Image

# Verificación de que las secciones 8 a 10 están en memoria
requeridas = ["tx_m", "art", "prod", "hist_val", "hist_t", "cand_val", "cand_t", "L_cf", "L_cf_t",
              "recientes", "L_hist_t", "clientes_eval", "clientes_t", "populares", "W_POP"]
faltantes = [v for v in requeridas if v not in globals()]
if faltantes:
    raise RuntimeError(f"Faltan variables de secciones anteriores: {faltantes}. Corra las secciones 2 a 10.")
if not os.environ.get("KAGGLE_API_TOKEN"):
    raise RuntimeError("No hay token de Kaggle en la sesión: configure el secret KAGGLE_API_TOKEN y corra la celda de credenciales.")

from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()

COMP = "h-and-m-personalized-fashion-recommendations"
DIR_IMG = f"{RUTA}/imagenes"
os.makedirs(DIR_IMG, exist_ok=True)

# ------------------------------------------------------------
# Productos que requieren imagen
# ------------------------------------------------------------
pop100_val = populares(hist_val, corte_val, W_POP, cand_val, n=100)
pop100_t   = populares(hist_t, corte_test, W_POP, cand_t, n=100)
origenes = {
    "Populares (100, validación y prueba)": set(pop100_val) | set(pop100_t),
    "Listas del CF (top 50, validación y prueba)": set().union(*L_cf.values(), *L_cf_t.values()),
    "Compras recientes de los evaluados (20)": set().union(
        *[recientes.get(c, []) for c in clientes_eval],
        *[L_hist_t.get(c, [])[:20] for c in clientes_t]),
}
necesarios = sorted({int(p) for p in set().union(*origenes.values())})
for nombre, s in origenes.items():
    print(f"{nombre:45s}: {len(s):,} productos")
print(f"{'Total (unión)':45s}: {len(necesarios):,} productos")

# Artículo representativo: color más vendido en la muestra; respaldo: primer artículo del catálogo
mas_vendido = (tx_m.groupby(["product_code", "article_id"]).size().rename("n").reset_index()
                   .sort_values("n", ascending=False).drop_duplicates("product_code"))
rep_dict = {int(p): int(a) for p, a in zip(mas_vendido["product_code"], mas_vendido["article_id"])}
for p, a in art.groupby("product_code")["article_id"].min().items():
    rep_dict.setdefault(int(p), int(a))
alternativos = art.groupby("product_code")["article_id"].apply(list).to_dict()

# ------------------------------------------------------------
# Descarga con pausas crecientes ante rechazos
# ------------------------------------------------------------
def descargar(pc, aid=None, max_intentos=5):
    """Descarga una imagen; solo marca 'sin imagen' si el 404 persiste tras varios intentos."""
    pc = int(pc)
    destino = f"{DIR_IMG}/{pc}.jpg"
    if os.path.exists(destino):
        return pc, "existente"
    aid = str(aid if aid is not None else rep_dict[pc]).zfill(10)
    tmp = f"{DIR_IMG}/tmp_{pc}_{aid}"
    error = ""
    for intento in range(max_intentos):
        try:
            api.competition_download_file(COMP, f"images/{aid[:3]}/{aid}.jpg", path=tmp, quiet=True)
        except Exception as e:
            error = str(e)
            if "403" in error:
                shutil.rmtree(tmp, ignore_errors=True)
                return pc, f"error: {error[:150]}"
            time.sleep(2 ** intento)                               # 1, 2, 4, 8, 16 s
            continue
        for raiz, _, archivos in os.walk(tmp):
            for f in archivos:
                if f.endswith(".zip"):
                    with zipfile.ZipFile(os.path.join(raiz, f)) as z:
                        z.extractall(raiz)
                    os.remove(os.path.join(raiz, f))
        jpgs = [os.path.join(r_, f) for r_, _, fs in os.walk(tmp) for f in fs if f.endswith(".jpg")]
        if jpgs:
            os.replace(jpgs[0], destino)
            shutil.rmtree(tmp, ignore_errors=True)
            return pc, "descargada"
        time.sleep(1)
    shutil.rmtree(tmp, ignore_errors=True)
    return pc, "sin imagen" if "404" in error else f"error: {error[:150]}"

def resumen(estados):
    return pd.Series([e if not e.startswith("error") else "error" for e in estados]).value_counts()

HILOS = 8
ya = sum(os.path.exists(f"{DIR_IMG}/{p}.jpg") for p in necesarios)
print(f"\nImágenes ya disponibles: {ya:,}")

# Diagnóstico con un producto pendiente
pendientes = [p for p in necesarios if not os.path.exists(f"{DIR_IMG}/{p}.jpg")]
if pendientes:
    aid_d = str(rep_dict[pendientes[0]]).zfill(10)
    try:
        api.competition_download_file(COMP, f"images/{aid_d[:3]}/{aid_d}.jpg", path=f"{DIR_IMG}/diag", quiet=True)
        print("Diagnóstico: la descarga individual funciona.")
    except Exception as e:
        print("Diagnóstico, mensaje de Kaggle:", repr(e)[:400])
    shutil.rmtree(f"{DIR_IMG}/diag", ignore_errors=True)

# Prueba con 200 pendientes
t0 = time.time()
with cfut.ThreadPoolExecutor(max_workers=HILOS) as ex:
    res200 = list(ex.map(descargar, pendientes[:200]))
dur = time.time() - t0
est200 = resumen([e for _, e in res200])
print(f"\nPrueba de {len(res200)}: {dur:.1f} s ({len(res200) / dur:.2f} imágenes/s)")
print(est200)
errores = [e for _, e in res200 if e.startswith("error")]
if errores:
    print("Primer error:", errores[0])

# Descarga completa solo si la prueba funcionó
if len(res200) and (est200.get("descargada", 0) + est200.get("existente", 0)) >= 0.9 * len(res200):
    pendientes = [p for p in necesarios if not os.path.exists(f"{DIR_IMG}/{p}.jpg")]
    print(f"\nDescargando {len(pendientes):,} pendientes...")
    estado, t_ini = {}, time.time()
    for i in range(0, len(pendientes), 1000):
        with cfut.ThreadPoolExecutor(max_workers=HILOS) as ex:
            estado.update(dict(ex.map(descargar, pendientes[i:i + 1000])))
        print(f"  {i + min(1000, len(pendientes) - i):,}/{len(pendientes):,} | "
              f"descargadas {sum(e == 'descargada' for e in estado.values()):,} | "
              f"{(time.time() - t_ini) / 60:.1f} min")
    print(resumen(estado.values()))

    # Productos sin imagen: probar otros colores del mismo producto (pocos hilos)
    sin_img = [p for p, e in estado.items() if e == "sin imagen"]
    if sin_img:
        print(f"\nBuscando otro color para {len(sin_img):,} productos sin imagen...")
        def otro_color(pc):
            for a in alternativos.get(pc, []):
                if a != rep_dict[pc]:
                    _, e = descargar(pc, aid=a, max_intentos=2)
                    if e == "descargada":
                        return pc, "descargada (otro color)"
            return pc, "sin imagen"
        with cfut.ThreadPoolExecutor(max_workers=4) as ex:
            print(resumen(dict(ex.map(otro_color, sin_img)).values()))
elif len(res200):
    print("\nLa prueba no descargó lo suficiente: no se continúa. Pegue esta salida.")

# ------------------------------------------------------------
# Resultado final
# ------------------------------------------------------------
con_imagen = sorted(p for p in necesarios if os.path.exists(f"{DIR_IMG}/{p}.jpg"))
print(f"\nProductos con imagen: {len(con_imagen):,} de {len(necesarios):,} "
      f"({100 * len(con_imagen) / len(necesarios):.2f} %)")
pd.DataFrame({"product_code": con_imagen}).to_parquet(f"{RUTA}/productos_con_imagen.parquet", index=False)

In [ ]:
# ------------------------------------------------------------
# Diagnóstico y descarga con control de ritmo
# ------------------------------------------------------------
ya = [f for f in os.listdir(DIR_IMG) if f.endswith(".jpg")]
print(f"Imágenes realmente descargadas hasta ahora: {len(ya):,}")

# Mensaje exacto de Kaggle para un producto marcado como "sin imagen"
p_diag = next(p for p in necesarios if not os.path.exists(f"{DIR_IMG}/{p}.jpg"))
aid = str(rep_dict[p_diag]).zfill(10)
try:
    api.competition_download_file(COMP, f"images/{aid[:3]}/{aid}.jpg", path=f"{DIR_IMG}/diag", quiet=True)
    print("Diagnóstico: la descarga individual funciona ahora.")
except Exception as e:
    print("Diagnóstico, mensaje de Kaggle:", repr(e)[:400])
shutil.rmtree(f"{DIR_IMG}/diag", ignore_errors=True)

def descargar(pc, max_intentos=5):
    """Descarga con pausas crecientes; solo marca 'sin imagen' si el 404 persiste."""
    pc = int(pc)
    destino = f"{DIR_IMG}/{pc}.jpg"
    if os.path.exists(destino):
        return pc, "existente"
    aid = str(rep_dict[pc]).zfill(10)
    tmp = f"{DIR_IMG}/tmp_{pc}"
    error = ""
    for intento in range(max_intentos):
        try:
            api.competition_download_file(COMP, f"images/{aid[:3]}/{aid}.jpg", path=tmp, quiet=True)
        except Exception as e:
            error = str(e)
            if "403" in error:
                shutil.rmtree(tmp, ignore_errors=True)
                return pc, f"error: {error[:150]}"
            time.sleep(2 ** intento)                       # 1, 2, 4, 8, 16 s
            continue
        for raiz, _, archivos in os.walk(tmp):
            for f in archivos:
                if f.endswith(".zip"):
                    with zipfile.ZipFile(os.path.join(raiz, f)) as z:
                        z.extractall(raiz)
                    os.remove(os.path.join(raiz, f))
        jpgs = [os.path.join(r_, f) for r_, _, fs in os.walk(tmp) for f in fs if f.endswith(".jpg")]
        if jpgs:
            os.replace(jpgs[0], destino)
            shutil.rmtree(tmp, ignore_errors=True)
            return pc, "descargada"
        time.sleep(1)
    shutil.rmtree(tmp, ignore_errors=True)
    return pc, "sin imagen" if "404" in error else f"error: {error[:150]}"

# Prueba con 200 pendientes antes de continuar
HILOS = 8
pendientes = [p for p in necesarios if not os.path.exists(f"{DIR_IMG}/{p}.jpg")]
t0 = time.time()
with cfut.ThreadPoolExecutor(max_workers=HILOS) as ex:
    res200 = list(ex.map(descargar, pendientes[:200]))
dur = time.time() - t0
est200 = pd.Series([e if not e.startswith("error") else "error" for _, e in res200]).value_counts()
print(f"\nPrueba de 200: {dur:.1f} s ({200 / dur:.2f} imágenes/s)")
print(est200)

# Continuar solo si la prueba descargó la gran mayoría
if est200.get("descargada", 0) >= 180:
    pendientes = [p for p in necesarios if not os.path.exists(f"{DIR_IMG}/{p}.jpg")]
    print(f"\nContinuando con {len(pendientes):,} pendientes...")
    estado, t_ini = {}, time.time()
    for i in range(0, len(pendientes), 1000):
        with cfut.ThreadPoolExecutor(max_workers=HILOS) as ex:
            estado.update(dict(ex.map(descargar, pendientes[i:i + 1000])))
        n_ok = sum(e == "descargada" for e in estado.values())
        print(f"  {i + min(1000, len(pendientes) - i):,}/{len(pendientes):,} | descargadas {n_ok:,} | "
              f"{(time.time() - t_ini) / 60:.1f} min")
    print(pd.Series([e if not e.startswith("error") else "error" for e in estado.values()]).value_counts())
else:
    print("\nLa prueba no descargó lo suficiente: no se continúa. Pegue esta salida.")

con_imagen = sorted(p for p in necesarios if os.path.exists(f"{DIR_IMG}/{p}.jpg"))
print(f"\nProductos con imagen: {len(con_imagen):,} de {len(necesarios):,} "
      f"({100 * len(con_imagen) / len(necesarios):.2f} %)")
pd.DataFrame({"product_code": con_imagen}).to_parquet(f"{RUTA}/productos_con_imagen.parquet", index=False)

#### **11.2 Representación visual de los productos**

Cada imagen se transforma en un vector numérico (*embedding*) con el codificador de imágenes de **CLIP** (Radford et al., 2021), en su versión ViT-B/32. CLIP fue entrenado con cientos de millones de pares de imagen y texto para ubicar en un mismo espacio las imágenes y sus descripciones, por lo que sus vectores capturan atributos visuales de alto nivel como el tipo de prenda, el color, el estampado y el estilo. Se usa el modelo preentrenado, sin ajuste fino, y cada producto queda representado por un vector de 512 dimensiones normalizado a norma uno, de modo que el producto interno entre dos productos equivale a su similitud de coseno. Antes del cálculo completo se mide la velocidad con 256 imágenes, y el cálculo se guarda por partes para poder reanudarlo si la sesión se interrumpe.

In [ ]:
# ============================================================
# 11.2 REPRESENTACIÓN VISUAL DE LOS PRODUCTOS (CLIP)
# ============================================================

import torch
from transformers import CLIPModel, CLIPProcessor

MODELO_CLIP = "openai/clip-vit-base-patch32"
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(os.cpu_count())
modelo_clip = CLIPModel.from_pretrained(MODELO_CLIP).to(dispositivo).eval()
procesador_clip = CLIPProcessor.from_pretrained(MODELO_CLIP)
print(f"Dispositivo: {dispositivo} | Núcleos de CPU: {os.cpu_count()}")

def cargar_imagen(pc):
    """Abre la imagen con decodificación reducida (CLIP la lleva a 224 × 224)."""
    im = Image.open(f"{DIR_IMG}/{pc}.jpg")
    im.draft("RGB", (320, 320))
    return im.convert("RGB")

def embeber(productos, lote=64):
    """Embeddings de imagen de CLIP normalizados (una fila por producto)."""
    vectores = []
    for i in range(0, len(productos), lote):
        with cfut.ThreadPoolExecutor(max_workers=8) as ex:
            imgs = list(ex.map(cargar_imagen, productos[i:i + lote]))
        with torch.no_grad():
            x = procesador_clip(images=imgs, return_tensors="pt").to(dispositivo)
            out = modelo_clip.get_image_features(**x)
            emb = out if isinstance(out, torch.Tensor) else getattr(out, "image_embeds", out.pooler_output)
        vectores.append(torch.nn.functional.normalize(emb, dim=-1).cpu().numpy().astype(np.float32))
    return np.vstack(vectores)

# ------------------------------------------------------------
# Prueba de velocidad con 256 imágenes
# ------------------------------------------------------------
con_imagen = pd.read_parquet(f"{RUTA}/productos_con_imagen.parquet")["product_code"].astype(int).tolist()
t0 = time.time()
_ = embeber(con_imagen[:256])
vel_emb = 256 / (time.time() - t0)
print(f"Velocidad: {vel_emb:.1f} imágenes/s | Estimación para {len(con_imagen):,}: "
      f"{len(con_imagen) / vel_emb / 60:.1f} min")

# ------------------------------------------------------------
# Cálculo completo por partes de 2.000 (reanudable)
# ------------------------------------------------------------
DIR_EMB = f"{RUTA}/emb_partes"
os.makedirs(DIR_EMB, exist_ok=True)
PARTE = 2000
t0 = time.time()
for k, i in enumerate(range(0, len(con_imagen), PARTE)):
    archivo = f"{DIR_EMB}/parte_{k:03d}.npy"
    if os.path.exists(archivo):
        continue
    np.save(archivo, embeber(con_imagen[i:i + PARTE]))
    print(f"  {min(i + PARTE, len(con_imagen)):,}/{len(con_imagen):,} | {(time.time() - t0) / 60:.1f} min")

E_img = np.vstack([np.load(f"{DIR_EMB}/parte_{k:03d}.npy")
                   for k in range(int(np.ceil(len(con_imagen) / PARTE)))])
fila_img = {p: i for i, p in enumerate(con_imagen)}
np.save(f"{RUTA}/embeddings_clip.npy", E_img)
pd.DataFrame({"product_code": con_imagen}).to_parquet(f"{RUTA}/embeddings_clip_indice.parquet", index=False)
print(f"\nEmbeddings: {E_img.shape} | Normas (deben ser 1): "
      f"{np.linalg.norm(E_img, axis=1).min():.4f} – {np.linalg.norm(E_img, axis=1).max():.4f}")

# ------------------------------------------------------------
# Verificación cualitativa: vecinos visuales de algunos productos populares
# ------------------------------------------------------------
ejemplos = [p for p in pop100_val if p in fila_img][:4]
fig, axes = plt.subplots(len(ejemplos), 6, figsize=(13, 2.6 * len(ejemplos)))
for f_, p in enumerate(ejemplos):
    sims = E_img @ E_img[fila_img[p]]
    vecinos = [con_imagen[j] for j in np.argsort(-sims)[:6]]
    for c_, v in enumerate(vecinos):
        ax = axes[f_, c_]
        ax.imshow(cargar_imagen(v))
        ax.set_title(("Consulta: " if c_ == 0 else f"{sims[fila_img[v]]:.2f} · ")
                     + str(prod["prod_name"].get(v, v))[:14], fontsize=7)
        ax.axis("off")
plt.suptitle("Productos visualmente más similares según CLIP (primera columna: producto consultado)")
plt.tight_layout()
plt.show()

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/proyecto_hm && cp {RUTA}/embeddings_clip.npy {RUTA}/embeddings_clip_indice.parquet /content/drive/MyDrive/proyecto_hm/

### **12. Fortalezas, debilidades e idoneidad de los métodos**

#### **12.1 Métodos de segmentación**

| Método | Fortalezas | Debilidades | Idoneidad en este caso |
|---|---|---|---|
| ACP | Reduce 22 variables a 9 componentes (73,6 % de la varianza) interpretables como intensidad, momento, valor y preferencia de compra | Los criterios de Kaiser y del 80 % no coinciden; captura solo relaciones lineales | Adecuado: la segmentación es insensible a usar 9 u 11 componentes (ARI de 0,996) |
| K-medias | Rápido y estable (ARI de 1,000 entre semillas); mejor silueta (0,192) | Segmentos definidos por la actividad y no por los gustos; empeora la recomendación (−14 % en validación) | Útil para describir la base de clientes, no para recomendar |
| K-medoides (CLARA) | Grupos equilibrados y diferenciados por línea de producto; robusto a atípicos | Más lento (151 s en la exploración) y menor silueta (0,162) | La segmentación más útil para recomendar entre las evaluadas, aunque su aporte no es significativo |
| Jerárquico (Ward) | No exige fijar $k$; el dendrograma orienta su elección | Costo $O(n^2)$: solo aplicable a una submuestra; grupos muy desbalanceados | Apoyo para elegir $k$, no como modelo final |
| Mezclas gaussianas | Asignación probabilística | BIC y AIC sin mínimo; peor calidad interna (silueta 0,098); el más lento (177 s) | No adecuado: modela la forma de la distribución, no grupos |
| DBSCAN | Identifica atípicos sin fijar $k$ | La densidad es continua y no hay grupos separados; alto consumo de memoria con $\varepsilon$ grande | Útil solo como detector de anomalías (0,3 % de clientes) |

#### **12.2 Métodos de recomendación**

| Método | Fortalezas | Debilidades | Idoneidad en este caso |
|---|---|---|---|
| Popularidad | Simple, sin historial requerido, respaldo para clientes nuevos | No personaliza; cobertura de 0,06 %; sensible a la semana (−18,9 % de validación a prueba) | Línea base y respaldo indispensable |
| Historial del cliente | Mejor recomendador individual en ambas semanas; estable | Solo captura recompras (85,5 % de sus aciertos); no sirve para clientes sin historia | Muy adecuado: la recompra es la señal más predictiva del dominio |
| Popularidad por segmento | Resuelve el arranque en frío por edad | Mejora no significativa sobre la popularidad global | Marginal |
| Reglas de asociación | Reglas interpretables (piezas de un mismo conjunto); rápido (6 s) | Solo 128 reglas; recomienda complementos que el cliente suele haber comprado ya | No adecuado como recomendador principal |
| Filtrado colaborativo | Personaliza y acierta productos nuevos (425 en prueba); estable | Exige historia reciente (+41 % al usar 12 semanas); la mayoría de sus aciertos son recompras | El método colaborativo más adecuado, condicionado a usar solo historia reciente |
| SVD | Rápido (11 s) | Captura solo el 18 % de la estructura de la matriz; cobertura baja; ventaja pequeña e inestable | Poco adecuado con datos tan dispersos |
| Contenido | No requiere otros clientes; cobertura alta (44–46 %) | Ignora la popularidad y las tendencias; recomienda productos similares pero poco comprados | No adecuado en un mercado dominado por tendencias |
| RBM | Aprende patrones no lineales de coocurrencia | El peor desempeño (−54 % de validación a prueba); cerca de 7 minutos por configuración; el error de reconstrucción no discrimina entre modelos | No adecuado con esta configuración |
| Híbrido ponderado | Mejor desempeño en validación; integra varias señales | Sus pesos se sobreajustan a la semana de calibración; en prueba empata con el historial | Equivalente al historial; requiere otro esquema de fusión para aportar valor |

#### **11.3 Lecciones transversales**

**La vigencia de la información domina.** La popularidad funciona mejor con una semana que con doce, y el filtrado colaborativo mejora un 41 % al restringirse a las últimas 12 semanas. En moda, los métodos que no ponderan la antigüedad de las compras (SVD, RBM y el filtrado colaborativo con toda la historia) quedan en desventaja frente a señales recientes, aunque sean más simples.

**La recompra es la señal más predictiva.** Aunque solo cerca del 8 % de las compras evaluadas son recompras (sección 4), entre el 65 % y el 86 % de los aciertos de los mejores modelos lo son, pues son las compras más fáciles de anticipar.

**La segmentación no supervisada aporta más a la comprensión de los clientes que a la recomendación.** Los segmentos permiten describir la base de clientes, y los de K-medoides capturan diferencias reales de preferencia, pero ninguno mejora de forma significativa la popularidad global como base de recomendación. Esto es coherente con Sarwar et al. (2002), quienes reportan que formar vecindarios a partir de clústeres gana escalabilidad a costa de precisión frente a vecindarios individuales.

**El diseño de la evaluación evitó conclusiones erróneas.** Sin la semana de prueba independiente, el híbrido habría parecido superior al historial; sin los intervalos *bootstrap* pareados, las mejoras de los segmentos (de 1 % a 3 %) habrían parecido relevantes. La partición temporal, la comparación pareada y la separación entre calibración y evaluación son, por tanto, parte esencial de los resultados.

### **13. Conclusiones y limitaciones**

#### **13.1 Conclusiones**

Este proyecto comparó, sobre una muestra estratificada del 10 % de los clientes de H&M y con una evaluación temporal en dos semanas independientes, cinco métodos de segmentación no supervisada y ocho recomendadores, y los combinó en un modelo híbrido. En respuesta a la pregunta de investigación, el recomendador híbrido mejora de forma significativa a la popularidad global (casi tres veces su MAP@12 en la semana de prueba), pero no supera al historial del cliente, que por sí solo alcanza el mismo desempeño. La mejora del híbrido observada en validación no se reprodujo en prueba, lo que evidencia el sobreajuste de la calibración de sus pesos.

Entre los métodos evaluados, solo el historial del cliente y el filtrado colaborativo basado en usuarios con historia reciente superan de forma sustancial y estable a la popularidad. La segmentación no supervisada permitió caracterizar a los clientes, en particular mediante K-medoides, que distingue preferencias por línea de producto, pero no mejoró significativamente la recomendación. El filtrado basado en contenido y las máquinas de Boltzmann restringidas resultaron inferiores a la popularidad, y las mezclas gaussianas y DBSCAN no identificaron grupos útiles en una nube de clientes de densidad continua.

El hallazgo central es que, en la recomendación de moda, la vigencia de la información y la recompra pesan más que la sofisticación del método: las señales simples y recientes superan a los modelos que tratan por igual toda la historia de compra.

#### **13.2 Limitaciones**

- **Una sola semana de prueba.** Los resultados corresponden a una semana de septiembre y podrían variar en otras temporadas; la caída de los modelos de popularidad entre validación y prueba sugiere que el desempeño depende de la semana evaluada.
- **Nivel de producto.** Al recomendar a nivel de `product_code`, recomprar el mismo producto en otro color cuenta como acierto, lo que puede favorecer a los métodos basados en el historial frente a una evaluación a nivel de artículo.
- **Esquema de fusión.** La fusión por rango recíproco con los pesos seleccionados ubica siempre el historial en las primeras posiciones, lo que impide aprovechar los aciertos en productos nuevos del filtrado colaborativo y de la popularidad. La constante de fusión se fijó en 60 sin calibrarla.
- **Ventanas temporales.** El SVD y la RBM se entrenaron con toda la historia; restringirlos a las semanas recientes, como se hizo con el filtrado colaborativo, podría mejorar su desempeño, pero no se evaluó por su costo computacional en el caso de la RBM.
- **Información disponible.** La base solo registra compras: no hay productos vistos sin comprar, devoluciones ni precios en moneda, y los códigos postales están anonimizados, lo que impidió aplicar la estimación de densidad con distancia de Haversine.
- **Evaluación sobre compradores.** Las métricas se calculan solo para los clientes que compraron en la semana evaluada; el desempeño sobre la base completa de clientes no es observable.

#### **13.3 Trabajo futuro**

- Evaluar un **híbrido mixto** que reserve una parte fija de la lista para productos nuevos (por ejemplo, seis posiciones para el historial y seis para el filtrado colaborativo o la popularidad), calibrado en validación.
- Ponderar las compras por su antigüedad en el filtrado colaborativo, el SVD y la RBM, en lugar de truncar la historia.
- Repetir la evaluación sobre varias semanas consecutivas para estimar la variabilidad del desempeño entre semanas.
- Incorporar la información visual de los productos, disponible en el concurso original, como señal adicional de contenido.

### **Referencias**

- Burke, R. (2002). Hybrid recommender systems: Survey and experiments. *User Modeling and User-Adapted Interaction, 12*(4), 331–370.
- Cormack, G. V., Clarke, C. L. A., & Büttcher, S. (2009). Reciprocal rank fusion outperforms Condorcet and individual rank learning methods. En *Proceedings of the 32nd International ACM SIGIR Conference on Research and Development in Information Retrieval* (pp. 758–759).
- Cremonesi, P., Koren, Y., & Turrin, R. (2010). Performance of recommender algorithms on top-N recommendation tasks. En *Proceedings of the Fourth ACM Conference on Recommender Systems* (pp. 39–46).
- Dacrema, M. F., Cremonesi, P., & Jannach, D. (2019). Are we really making much progress? A worrying analysis of recent neural recommendation approaches. En *Proceedings of the 13th ACM Conference on Recommender Systems* (pp. 101–109).
- Fader, P. S., Hardie, B. G. S., & Lee, K. L. (2005). RFM and CLV: Using iso-value curves for customer base analysis. *Journal of Marketing Research, 42*(4), 415–430.
- H&M Group. (2022). *H&M Personalized Fashion Recommendations* [Conjunto de datos]. Kaggle.
- Hu, Y., Koren, Y., & Volinsky, C. (2008). Collaborative filtering for implicit feedback datasets. En *2008 Eighth IEEE International Conference on Data Mining* (pp. 263–272).
- Ji, Y., Sun, A., Zhang, J., & Li, C. (2023). A critical study on data leakage in recommender system offline evaluation. *ACM Transactions on Information Systems, 41*(3).
- Kaufman, L., & Rousseeuw, P. J. (1990). *Finding groups in data: An introduction to cluster analysis*. Wiley.
- Mwiti, D. (s. f.). *H&M Personalized Fashion Recommendations* (versión de bajo consumo de memoria) [Conjunto de datos]. Kaggle.
- Salakhutdinov, R., Mnih, A., & Hinton, G. (2007). Restricted Boltzmann machines for collaborative filtering. En *Proceedings of the 24th International Conference on Machine Learning* (pp. 791–798).
- Särndal, C.-E., Swensson, B., & Wretman, J. (1992). *Model assisted survey sampling*. Springer.
- Sarwar, B. M., Karypis, G., Konstan, J., & Riedl, J. (2002). Recommender systems for large-scale e-commerce: Scalable neighborhood formation using clustering. En *Proceedings of the Fifth International Conference on Computer and Information Technology*.
- Schubert, E., & Rousseeuw, P. J. (2021). Fast and eager k-medoids clustering: O(k) runtime improvement of the PAM, CLARA, and CLARANS algorithms. *Information Systems, 101*, 101804.